In [1]:
# Mount your Google Drive on Colab if your are loading the dataset file from your Drive
# upload data to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Download TechSolve - Ticket Data dataset file from Google Drive

In [2]:
import pandas as pd

file_path = "drive/MyDrive/TechSolve_Ticket_Data/TechSolve - Ticket Data.xlsx"
excel_file = pd.ExcelFile(file_path, engine="openpyxl")
print(excel_file.sheet_names)


# Read/copy data from the sheet named '2025 Lotto Results'
df = pd.read_excel(
    file_path,
    sheet_name="Sheet1",

)

# Make a copy of the data
df = df.copy()

# Preview the copied data
print(df.tail())

['Sheet1']
        ticket_id customer_id    customer_name                customer_email  \
100846     100847   ACC-06526  Charles Johnson  charles.johnson457@yahoo.com   
100847     100848   ACC-03650    William Brown  william.brown671@hotmail.com   
100848     100849   ACC-01465   Charles Taylor   charles.taylor867@gmail.com   
100849     100850   ACC-03432     Susan Thomas     susan.thomas126@gmail.com   
100850     100851   ACC-06124    William Lopez  william.lopez932@hotmail.com   

             company_name account_type customer_segment  \
100846                NaN  Residential       Individual   
100847  Pioneer Logistics     Business   Small Business   
100848                NaN  Residential       Individual   
100849                NaN  Residential   Small Business   
100850   Island Transport     Business        Corporate   

                     industry        billing_contact_email account_manager  \
100846                    NaN                          NaN             NaN 

# Download External Data

In [3]:
#!/usr/bin/env python3
"""
Download and prepare New Zealand public-holiday data for the TechSolve
ticket dataset.

Period
------
1 January 2023 to 31 December 2025

Analytical join key
-------------------
DateRegionKey = YYYYMMDD|Region

For example:
    20240715|Auckland

Primary output dataframes
-------------------------
holidays_df
tickets_df: add TicketDate, RegionAnalytics, DateKey, DateRegionKey for each ticket.

Output files
------------
nz_holiday_calendar_2023_2025.csv
techsolve_tickets_with_date_region_key.csv
download_manifest.json
"""

from __future__ import annotations

import json
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

START_DATE = date(2023, 1, 1)
END_DATE = date(2025, 12, 31)

FILE_PATH = (
    "/content/drive/MyDrive/"
    "TechSolve_Ticket_Data"
)

OUTPUT_DIRECTORY = Path(FILE_PATH)

OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

TICKET_FILE = (
    OUTPUT_DIRECTORY
    / "TechSolve - Ticket Data.xlsx"
)

TICKET_SHEET = "Sheet1"

NAGER_URL = (
    "https://date.nager.at/api/v3/"
    "PublicHolidays/{year}/NZ"
)

USER_AGENT = (
    "TechSolve-PowerBI-Holiday-Enrichment/1.0"
)


# Regions used in the TechSolve ticket dataset.
TICKET_REGIONS = [
    "Auckland",
    "Bay of Plenty",
    "Canterbury",
    "Hawke's Bay",
    "Manawatu-Whanganui",
    "Nelson-Marlborough",
    "Northland",
    "Otago",
    "Southland",
    "Taranaki",
    "Waikato",
    "Wellington",
]


print(
    f"Output folder: {OUTPUT_DIRECTORY}"
)

print(
    f"Ticket workbook: {TICKET_FILE}"
)

print(
    f"Ticket workbook exists: "
    f"{TICKET_FILE.exists()}"
)


# =============================================================================
# 2. GENERIC HELPERS
# =============================================================================

def normalise_region(
    value: Any,
) -> Optional:
    """
    Standardise region names so the external holiday data can join to the
    TechSolve ticket dataframe.

    Missing values return None.
    """

    if pd.isna(value):
        return None

    text = str(value).strip()

    aliases = {
        "auckland region": "Auckland",
        "bay of plenty region": "Bay of Plenty",
        "canterbury region": "Canterbury",
        "hawkes bay": "Hawke's Bay",
        "hawke’s bay": "Hawke's Bay",
        "hawke's bay region": "Hawke's Bay",
        "manawatu whanganui": "Manawatu-Whanganui",
        "manawatū-whanganui": "Manawatu-Whanganui",
        "manawatu-wanganui": "Manawatu-Whanganui",
        "nelson marlborough": "Nelson-Marlborough",
        "nelson-marlborough region": "Nelson-Marlborough",
        "northland region": "Northland",
        "otago region": "Otago",
        "southland region": "Southland",
        "taranaki region": "Taranaki",
        "waikato region": "Waikato",
        "wellington region": "Wellington",
    }

    return aliases.get(
        text.lower(),
        text,
    )


def create_date_region_key(
    date_series: pd.Series,
    region_series: pd.Series,
) -> pd.Series:
    """
    Create the analytical key:

        YYYYMMDD|Region

    Example:

        20240715|Auckland

    Missing dates or regions produce pandas NA.
    """

    parsed_dates = pd.to_datetime(
        date_series,
        errors="coerce",
    )

    normalised_regions = (
        region_series
        .map(normalise_region)
        .astype("string")
    )

    date_text = (
        parsed_dates
        .dt.strftime("%Y%m%d")
        .astype("string")
    )

    valid = (
        parsed_dates.notna()
        & normalised_regions.notna()
        & normalised_regions.str.strip().ne("")
    )

    output = pd.Series(
        pd.NA,
        index=date_series.index,
        dtype="string",
    )

    output.loc[valid] = (
        date_text.loc[valid]
        + "|"
        + normalised_regions.loc[valid]
    )

    return output


def request_json(
    url: str,
    attempts: int = 4,
    timeout_seconds: int = 90,
) -> Any:
    """
    Download JSON with retry handling.
    """

    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/json",
    }

    last_error: Optional[Exception] = None

    for attempt_number in range(
        1,
        attempts + 1,
    ):
        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=timeout_seconds,
            )

            response.raise_for_status()

            return response.json()

        except requests.RequestException as exc:
            last_error = exc

        except ValueError as exc:
            last_error = ValueError(
                f"The response from {url} "
                "was not valid JSON."
            )

        if attempt_number < attempts:
            sleep_seconds = (
                2 ** (attempt_number - 1)
            )

            print(
                f"Request failed. Retrying in "
                f"{sleep_seconds} second(s): "
                f"{url}"
            )

            time.sleep(
                sleep_seconds
            )

    raise RuntimeError(
        f"Unable to download holiday data after "
        f"{attempts} attempts: {url}"
    ) from last_error


# =============================================================================
# 3. DATE HELPERS
# =============================================================================

def nearest_monday(
    calendar_date: date,
) -> date:
    """
    Return the Monday nearest to a calendar date.

    If the previous and following Mondays are equally close, the following
    Monday is used.
    """

    previous_monday = (
        calendar_date
        - timedelta(
            days=calendar_date.weekday()
        )
    )

    following_monday = (
        previous_monday
        + timedelta(days=7)
    )

    distance_to_previous = (
        calendar_date
        - previous_monday
    )

    distance_to_following = (
        following_monday
        - calendar_date
    )

    if (
        distance_to_previous
        < distance_to_following
    ):
        return previous_monday

    return following_monday


def nth_weekday(
    year: int,
    month: int,
    weekday: int,
    occurrence: int,
) -> date:
    """
    Return the nth weekday in a month.

    weekday values:
        Monday = 0
        Tuesday = 1
        Wednesday = 2
        Thursday = 3
        Friday = 4
        Saturday = 5
        Sunday = 6
    """

    first_day = date(
        year,
        month,
        1,
    )

    offset = (
        weekday
        - first_day.weekday()
    ) % 7

    first_occurrence = (
        first_day
        + timedelta(days=offset)
    )

    return (
        first_occurrence
        + timedelta(
            days=7 * (occurrence - 1)
        )
    )


def easter_sunday(
    year: int,
) -> date:
    """
    Calculate Gregorian Easter Sunday using the
    Meeus/Jones/Butcher algorithm.
    """

    a = year % 19
    b, c = divmod(year, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (
        19 * a
        + b
        - d
        - g
        + 15
    ) % 30

    i, k = divmod(c, 4)

    l = (
        32
        + 2 * e
        + 2 * i
        - h
        - k
    ) % 7

    m = (
        a
        + 11 * h
        + 22 * l
    ) // 451

    month = (
        h
        + l
        - 7 * m
        + 114
    ) // 31

    day = (
        (
            h
            + l
            - 7 * m
            + 114
        )
        % 31
    ) + 1

    return date(
        year,
        month,
        day,
    )


# =============================================================================
# 4. REGIONAL ANNIVERSARY DAYS
# =============================================================================

def regional_anniversary_events(
    year: int,
) -> List[Dict[str, Any]]:
    """
    Return regional anniversary observance dates for the regions represented
    in the TechSolve ticket dataset.

    These records are created independently of the public-holiday API so that
    a regional anniversary is never expanded to every region.

    Notes
    -----
    The combined Nelson-Marlborough and Manawatu-Whanganui regions require
    analytical mapping assumptions because their boundaries do not correspond
    perfectly with every local anniversary observance.
    """

    rows: List[Dict[str, Any]] = []

    def add(
        region: str,
        event_date: date,
        event_name: str,
    ) -> None:
        if region not in TICKET_REGIONS:
            raise ValueError(
                f"Regional holiday contains an unsupported region: "
                f"{region}"
            )

        rows.append(
            {
                "HolidayEventDate": event_date,
                "HolidayName": event_name,
                "HolidayScope": "Regional",
                "Region": region,
                "HolidaySource": (
                    "Calculated regional anniversary rule"
                ),
            }
        )

    # -----------------------------------------------------------------
    # Auckland Anniversary Day
    # Monday nearest 29 January.
    # -----------------------------------------------------------------

    auckland_anniversary = nearest_monday(
        date(
            year,
            1,
            29,
        )
    )

    # These assignments are explicit. Add or remove regions here only
    # if your organisation's approved regional calendar requires it.
    for region in [
        "Auckland",
        "Northland",
        "Waikato",
        "Bay of Plenty",
    ]:
        add(
            region=region,
            event_date=auckland_anniversary,
            event_name="Auckland Anniversary Day",
        )

    # -----------------------------------------------------------------
    # Wellington Anniversary Day
    # Monday nearest 22 January.
    # -----------------------------------------------------------------

    wellington_anniversary = nearest_monday(
        date(
            year,
            1,
            22,
        )
    )

    for region in [
        "Wellington",
        "Manawatu-Whanganui",
    ]:
        add(
            region=region,
            event_date=wellington_anniversary,
            event_name="Wellington Anniversary Day",
        )

    # -----------------------------------------------------------------
    # Taranaki Anniversary Day
    # Second Monday in March.
    # -----------------------------------------------------------------

    add(
        region="Taranaki",
        event_date=nth_weekday(
            year=year,
            month=3,
            weekday=0,
            occurrence=2,
        ),
        event_name="Taranaki Anniversary Day",
    )

    # Labour Day is the fourth Monday in October.
    labour_day = nth_weekday(
        year=year,
        month=10,
        weekday=0,
        occurrence=4,
    )

    # -----------------------------------------------------------------
    # Hawke's Bay Anniversary Day
    # Friday before Labour Day.
    # -----------------------------------------------------------------

    add(
        region="Hawke's Bay",
        event_date=(
            labour_day
            - timedelta(days=3)
        ),
        event_name="Hawke's Bay Anniversary Day",
    )

    # -----------------------------------------------------------------
    # Marlborough Anniversary Day
    # Monday following Labour Day.
    #
    # Analytical assumption:
    # Applied to the combined Nelson-Marlborough ticket region.
    # -----------------------------------------------------------------

    add(
        region="Nelson-Marlborough",
        event_date=(
            labour_day
            + timedelta(days=7)
        ),
        event_name="Marlborough Anniversary Day",
    )

    # -----------------------------------------------------------------
    # Canterbury Anniversary / Show Day
    # Second Friday after the first Tuesday in November.
    # -----------------------------------------------------------------

    first_tuesday = nth_weekday(
        year=year,
        month=11,
        weekday=1,
        occurrence=1,
    )

    first_friday_after = (
        first_tuesday
        + timedelta(
            days=(
                4
                - first_tuesday.weekday()
            )
            % 7
        )
    )

    canterbury_show_day = (
        first_friday_after
        + timedelta(days=7)
    )

    add(
        region="Canterbury",
        event_date=canterbury_show_day,
        event_name="Canterbury Anniversary Day",
    )

    # -----------------------------------------------------------------
    # Otago Anniversary Day
    # Monday nearest 23 March.
    # -----------------------------------------------------------------

    add(
        region="Otago",
        event_date=nearest_monday(
            date(
                year,
                3,
                23,
            )
        ),
        event_name="Otago Anniversary Day",
    )

    # -----------------------------------------------------------------
    # Southland Anniversary Day
    # Tuesday after Easter Monday.
    # -----------------------------------------------------------------

    add(
        region="Southland",
        event_date=(
            easter_sunday(year)
            + timedelta(days=2)
        ),
        event_name="Southland Anniversary Day",
    )

    return rows

# =============================================================================
# 5. NATIONAL PUBLIC HOLIDAYS
# =============================================================================

def download_national_holiday_events(
    start_date: date,
    end_date: date,
) -> pd.DataFrame:
    """
    Download nationwide New Zealand public holidays.

    The API may contain both nationwide and regional holidays.

    Only records explicitly marked as global and without regional county codes
    are expanded to every TechSolve region.

    Regional anniversary days are created separately by
    regional_anniversary_events().
    """

    rows: List[Dict[str, Any]] = []

    excluded_api_holidays: List[
        Dict[str, Any]
    ] = []

    for year in range(
        start_date.year,
        end_date.year + 1,
    ):
        url = NAGER_URL.format(
            year=year
        )

        print(
            f"Downloading nationwide public holidays "
            f"for {year}..."
        )

        response_data = request_json(
            url
        )

        if not isinstance(
            response_data,
            list,
        ):
            raise ValueError(
                f"Unexpected holiday API response for "
                f"{year}. A list was expected."
            )

        for item in response_data:
            holiday_name = (
                item.get("localName")
                or item.get("name")
                or "Public Holiday"
            )

            is_global = item.get(
                "global"
            )

            counties = item.get(
                "counties"
            )

            # A genuinely nationwide holiday must:
            # 1. be explicitly marked global; and
            # 2. have no regional county codes.
            has_regional_counties = bool(
                counties
            )

            is_nationwide = (
                is_global is True
                and not has_regional_counties
            )

            if not is_nationwide:
                # Convert counties to immutable text now.
                # This avoids unhashable list errors later.
                if isinstance(
                    counties,
                    list,
                ):
                    counties_text = " | ".join(
                        sorted(
                            str(county)
                            for county in counties
                        )
                    )

                elif counties is None:
                    counties_text = ""

                else:
                    counties_text = str(
                        counties
                    )

                excluded_api_holidays.append(
                    {
                        "Year": year,
                        "HolidayName": holiday_name,
                        "Global": is_global,
                        "Counties": counties_text,
                    }
                )

                continue

            raw_date = item.get(
                "date"
            )

            if not raw_date:
                continue

            event_timestamp = pd.to_datetime(
                raw_date,
                errors="coerce",
            )

            if pd.isna(
                event_timestamp
            ):
                continue

            event_date = (
                event_timestamp.date()
            )

            if not (
                start_date
                <= event_date
                <= end_date
            ):
                continue

            # Expand genuine nationwide holidays to every region.
            for region in TICKET_REGIONS:
                rows.append(
                    {
                        "HolidayEventDate": event_date,
                        "HolidayName": holiday_name,
                        "HolidayScope": "National",
                        "Region": region,
                        "HolidaySource": (
                            "Nager.Date public API - "
                            "global nationwide record"
                        ),
                    }
                )

    national_events = pd.DataFrame(
        rows
    )

    if national_events.empty:
        raise ValueError(
            "No nationwide public holidays were returned. "
            "Inspect the API response fields 'global' "
            "and 'counties'."
        )

    # Ensure the nationwide result itself contains no duplicates.
    national_events = (
        national_events
        .drop_duplicates(
            subset=[
                "HolidayEventDate",
                "HolidayName",
                "HolidayScope",
                "Region",
            ]
        )
        .sort_values(
            [
                "HolidayEventDate",
                "Region",
                "HolidayName",
            ]
        )
        .reset_index(drop=True)
    )

    print()
    print(
        f"Nationwide holiday event-region rows created: "
        f"{len(national_events):,}"
    )

    if excluded_api_holidays:
        excluded_df = pd.DataFrame(
            excluded_api_holidays
        )

        # Counties is already text in this corrected function,
        # so drop_duplicates() is safe.
        excluded_df = (
            excluded_df
            .drop_duplicates(
                subset=[
                    "Year",
                    "HolidayName",
                    "Global",
                    "Counties",
                ]
            )
            .sort_values(
                [
                    "Year",
                    "HolidayName",
                ]
            )
            .reset_index(drop=True)
        )

        print(
            f"Regional or non-global API records excluded: "
            f"{len(excluded_df):,}"
        )

        print(
            "Excluded API holiday names:"
        )

        for holiday in sorted(
            excluded_df["HolidayName"]
            .dropna()
            .astype(str)
            .unique()
        ):
            print(
                f"  - {holiday}"
            )

    return national_events


# =============================================================================
# 6. CREATE THE HOLIDAY DATAFRAME
# =============================================================================

def create_holiday_dataframe(
    start_date: date,
    end_date: date,
) -> pd.DataFrame:
    """
    Create one row for every date and region.

    DateRegionKey is unique and can link directly to the same key on the
    TechSolve ticket dataframe.
    """

    national_events = (
        download_national_holiday_events(
            start_date=start_date,
            end_date=end_date,
        )
    )

    regional_rows: List[
        Dict[str, Any]
    ] = []

    for year in range(
        start_date.year,
        end_date.year + 1,
    ):
        regional_rows.extend(
            regional_anniversary_events(
                year
            )
        )

    regional_events = pd.DataFrame(
        regional_rows
    )

    events = pd.concat(
        [
            national_events,
            regional_events,
        ],
        ignore_index=True,
    )

    events["HolidayEventDate"] = pd.to_datetime(
        events["HolidayEventDate"],
        errors="coerce",
    )

    events["Region"] = (
        events["Region"]
        .map(normalise_region)
    )

    events = events.dropna(
        subset=[
            "HolidayEventDate",
            "Region",
        ]
    )

    events = events.loc[
        events["HolidayEventDate"].between(
            pd.Timestamp(start_date),
            pd.Timestamp(end_date),
        )
    ].copy()

    events = (
        events
        .drop_duplicates(
            subset=[
                "HolidayEventDate",
                "HolidayName",
                "HolidayScope",
                "Region",
            ]
        )
        .sort_values(
            [
                "HolidayEventDate",
                "Region",
                "HolidayScope",
                "HolidayName",
            ]
        )
        .reset_index(drop=True)
    )

    # Create one row for every date and region.
    calendar = pd.MultiIndex.from_product(
        [
            pd.date_range(
                start=start_date,
                end=end_date,
                freq="D",
            ),
            TICKET_REGIONS,
        ],
        names=[
            "Date",
            "Region",
        ],
    ).to_frame(
        index=False
    )

    calendar["Region"] = (
        calendar["Region"]
        .map(normalise_region)
    )

    # Combine multiple holidays occurring on the same date.
    same_day = (
        events
        .groupby(
            [
                "HolidayEventDate",
                "Region",
            ],
            as_index=False,
        )
        .agg(
            HolidayName=(
                "HolidayName",
                lambda values: " | ".join(
                    sorted(
                        set(values)
                    )
                ),
            ),
            HolidayScope=(
                "HolidayScope",
                lambda values: " | ".join(
                    sorted(
                        set(values)
                    )
                ),
            ),
            HolidaySource=(
                "HolidaySource",
                lambda values: " | ".join(
                    sorted(
                        set(values)
                    )
                ),
            ),
        )
        .rename(
            columns={
                "HolidayEventDate": "Date",
            }
        )
    )

    holidays_df = calendar.merge(
        same_day,
        on=[
            "Date",
            "Region",
        ],
        how="left",
        validate="one_to_one",
    )

    holidays_df["IsPublicHoliday"] = (
        holidays_df["HolidayName"]
        .notna()
        .astype("int8")
    )

    # Identify the day immediately before a holiday.
    before_lookup = same_day.copy()

    before_lookup["Date"] = (
        before_lookup["Date"]
        - pd.Timedelta(days=1)
    )

    before_lookup = before_lookup.rename(
        columns={
            "HolidayName": (
                "FollowingHolidayName"
            ),
            "HolidayScope": (
                "FollowingHolidayScope"
            ),
        }
    )[
        [
            "Date",
            "Region",
            "FollowingHolidayName",
            "FollowingHolidayScope",
        ]
    ]

    holidays_df = holidays_df.merge(
        before_lookup,
        on=[
            "Date",
            "Region",
        ],
        how="left",
        validate="one_to_one",
    )

    holidays_df["IsDayBeforeHoliday"] = (
        holidays_df[
            "FollowingHolidayName"
        ]
        .notna()
        .astype("int8")
    )

    # Identify the day immediately after a holiday.
    after_lookup = same_day.copy()

    after_lookup["Date"] = (
        after_lookup["Date"]
        + pd.Timedelta(days=1)
    )

    after_lookup = after_lookup.rename(
        columns={
            "HolidayName": (
                "PreviousHolidayName"
            ),
            "HolidayScope": (
                "PreviousHolidayScope"
            ),
        }
    )[
        [
            "Date",
            "Region",
            "PreviousHolidayName",
            "PreviousHolidayScope",
        ]
    ]

    holidays_df = holidays_df.merge(
        after_lookup,
        on=[
            "Date",
            "Region",
        ],
        how="left",
        validate="one_to_one",
    )

    holidays_df["IsDayAfterHoliday"] = (
        holidays_df[
            "PreviousHolidayName"
        ]
        .notna()
        .astype("int8")
    )

    holidays_df["IsWeekend"] = (
        holidays_df["Date"]
        .dt.weekday
        .ge(5)
        .astype("int8")
    )

    holidays_df["IsBusinessDay"] = (
        (
            holidays_df[
                "IsWeekend"
            ].eq(0)
            & holidays_df[
                "IsPublicHoliday"
            ].eq(0)
        )
        .astype("int8")
    )

    holidays_df["DayName"] = (
        holidays_df["Date"]
        .dt.day_name()
    )

    holidays_df["DayOfWeekNumber"] = (
        holidays_df["Date"]
        .dt.weekday
        .add(1)
        .astype("int8")
    )

    holidays_df["Year"] = (
        holidays_df["Date"]
        .dt.year
        .astype("int16")
    )

    holidays_df["Quarter"] = (
        "Q"
        + holidays_df["Date"]
        .dt.quarter
        .astype(str)
    )

    holidays_df["MonthNumber"] = (
        holidays_df["Date"]
        .dt.month
        .astype("int8")
    )

    holidays_df["MonthName"] = (
        holidays_df["Date"]
        .dt.month_name()
    )

    holidays_df["YearMonth"] = (
        holidays_df["Date"]
        .dt.strftime("%Y-%m")
    )

    holidays_df["YearMonthSort"] = (
        holidays_df["Date"]
        .dt.strftime("%Y%m")
        .astype("int32")
    )

    holidays_df["DateKey"] = (
        holidays_df["Date"]
        .dt.strftime("%Y%m%d")
        .astype("string")
    )

    holidays_df["DateRegionKey"] = (
        create_date_region_key(
            holidays_df["Date"],
            holidays_df["Region"],
        )
    )

    preferred_columns = [
        "DateRegionKey",
        "DateKey",
        "Date",
        "Region",
        "Year",
        "Quarter",
        "MonthNumber",
        "MonthName",
        "YearMonth",
        "YearMonthSort",
        "DayOfWeekNumber",
        "DayName",
        "HolidayName",
        "HolidayScope",
        "HolidaySource",
        "IsPublicHoliday",
        "FollowingHolidayName",
        "FollowingHolidayScope",
        "IsDayBeforeHoliday",
        "PreviousHolidayName",
        "PreviousHolidayScope",
        "IsDayAfterHoliday",
        "IsWeekend",
        "IsBusinessDay",
    ]

    holidays_df = holidays_df[
        preferred_columns
    ]

    holidays_df = (
        holidays_df
        .sort_values(
            [
                "Date",
                "Region",
            ]
        )
        .reset_index(drop=True)
    )

    return holidays_df


# =============================================================================
# 7. PREPARE THE TECHSOLVE TICKET DATAFRAME
# =============================================================================

def prepare_ticket_dataframe(
    source_file: Path,
    sheet_name: str,
) -> pd.DataFrame:
    """
    Read the TechSolve workbook and add:

    - TicketDate
    - RegionAnalytics
    - DateKey
    - DateRegionKey
    """

    if not source_file.exists():
        raise FileNotFoundError(
            f"Ticket workbook was not found: "
            f"{source_file}"
        )

    tickets_df = pd.read_excel(
        source_file,
        sheet_name=sheet_name,
        engine="openpyxl",
    )

    required_columns = {
        "ticket_created_date",
        "region",
    }

    missing_columns = (
        required_columns
        .difference(
            tickets_df.columns
        )
    )

    if missing_columns:
        raise ValueError(
            "Ticket dataframe is missing "
            f"required fields: "
            f"{sorted(missing_columns)}"
        )

    tickets_df[
        "ticket_created_date"
    ] = pd.to_datetime(
        tickets_df[
            "ticket_created_date"
        ],
        errors="coerce",
    )

    tickets_df["TicketDate"] = (
        tickets_df[
            "ticket_created_date"
        ]
        .dt.normalize()
    )

    tickets_df["RegionAnalytics"] = (
        tickets_df["region"]
        .map(normalise_region)
    )

    tickets_df["DateKey"] = (
        tickets_df["TicketDate"]
        .dt.strftime("%Y%m%d")
        .astype("string")
    )

    tickets_df["DateRegionKey"] = (
        create_date_region_key(
            tickets_df["TicketDate"],
            tickets_df["RegionAnalytics"],
        )
    )

    return tickets_df


# =============================================================================
# 8. VALIDATION
# =============================================================================

def validate_holiday_dataframe(
    holidays_df: pd.DataFrame,
) -> None:
    """
    Validate the holiday calendar before saving it.
    """

    required_columns = {
        "DateRegionKey",
        "DateKey",
        "Date",
        "Region",
        "HolidayName",
        "IsPublicHoliday",
        "IsDayBeforeHoliday",
        "IsDayAfterHoliday",
        "IsWeekend",
        "IsBusinessDay",
    }

    missing_columns = (
        required_columns
        .difference(
            holidays_df.columns
        )
    )

    if missing_columns:
        raise ValueError(
            "Holiday dataframe is missing fields: "
            f"{sorted(missing_columns)}"
        )

    if holidays_df.empty:
        raise ValueError(
            "Holiday dataframe is empty."
        )

    if holidays_df[
        "DateRegionKey"
    ].isna().any():
        raise ValueError(
            "Holiday dataframe contains a "
            "missing DateRegionKey."
        )

    duplicate_keys = int(
        holidays_df[
            "DateRegionKey"
        ]
        .duplicated()
        .sum()
    )

    if duplicate_keys:
        raise ValueError(
            f"Holiday dataframe contains "
            f"{duplicate_keys:,} duplicate "
            "DateRegionKey values."
        )

    expected_date_count = len(
        pd.date_range(
            START_DATE,
            END_DATE,
            freq="D",
        )
    )

    expected_rows = (
        expected_date_count
        * len(TICKET_REGIONS)
    )

    if len(holidays_df) != expected_rows:
        raise ValueError(
            f"Holiday dataframe has "
            f"{len(holidays_df):,} rows; "
            f"{expected_rows:,} were expected."
        )


def validate_regional_holiday_assignments(
    holidays_df: pd.DataFrame,
) -> None:
    """
    Ensure that regional anniversary holidays are assigned only to explicitly
    permitted TechSolve regions.

    The validation stops the programme if a regional holiday leaks into an
    unrelated region.
    """

    permitted_regions = {
        "Auckland Anniversary Day": {
            "Auckland",
            "Northland",
            "Waikato",
            "Bay of Plenty",
        },

        "Wellington Anniversary Day": {
            "Wellington",
            "Manawatu-Whanganui",
        },

        "Taranaki Anniversary Day": {
            "Taranaki",
        },

        "Hawke's Bay Anniversary Day": {
            "Hawke's Bay",
        },

        "Marlborough Anniversary Day": {
            "Nelson-Marlborough",
        },

        "Canterbury Anniversary Day": {
            "Canterbury",
        },

        "Otago Anniversary Day": {
            "Otago",
        },

        "Southland Anniversary Day": {
            "Southland",
        },
    }

    invalid_frames: List[pd.DataFrame] = []

    for (
        holiday_name,
        allowed_regions,
    ) in permitted_regions.items():
        holiday_match = (
            holidays_df["HolidayName"]
            .astype("string")
            .str.contains(
                holiday_name,
                case=False,
                na=False,
                regex=False,
            )
        )

        region_not_allowed = (
            ~holidays_df["Region"]
            .isin(allowed_regions)
        )

        invalid_rows = holidays_df.loc[
            holiday_match
            & region_not_allowed,
            [
                "DateRegionKey",
                "Date",
                "Region",
                "HolidayName",
                "HolidayScope",
                "HolidaySource",
            ],
        ].copy()

        if not invalid_rows.empty:
            invalid_frames.append(
                invalid_rows
            )

    if invalid_frames:
        invalid_assignments = pd.concat(
            invalid_frames,
            ignore_index=True,
        )

        print()
        print(
            "Incorrect regional holiday assignments detected:"
        )

        print(
            invalid_assignments.to_string(
                index=False
            )
        )

        raise ValueError(
            "Regional anniversary holidays were assigned "
            "to unrelated regions. The holiday output was not saved."
        )

    print(
        "Regional holiday assignments validated successfully."
    )
   
   
def validate_known_holiday_examples(
    holidays_df: pd.DataFrame,
) -> None:
    """
    Validate several known date-region outcomes.

    These checks specifically prevent the defect visible in the attached
    output, where Wellington Anniversary Day was assigned across all regions.
    """

    expected_results = {
        # Auckland is not observing Wellington Anniversary Day.
        "20230123|Auckland": {
            "HolidayName": None,
            "IsPublicHoliday": 0,
        },

        # Wellington is observing Wellington Anniversary Day.
        "20230123|Wellington": {
            "HolidayName": "Wellington Anniversary Day",
            "IsPublicHoliday": 1,
        },

        # Auckland Anniversary Day.
        "20230130|Auckland": {
            "HolidayName": "Auckland Anniversary Day",
            "IsPublicHoliday": 1,
        },

        # Auckland is not observing Wellington Anniversary Day in 2024.
        "20240122|Auckland": {
            "HolidayName": None,
            "IsPublicHoliday": 0,
        },

        # Wellington Anniversary Day in 2024.
        "20240122|Wellington": {
            "HolidayName": "Wellington Anniversary Day",
            "IsPublicHoliday": 1,
        },

        # Auckland is not observing Wellington Anniversary Day in 2025.
        "20250120|Auckland": {
            "HolidayName": None,
            "IsPublicHoliday": 0,
        },

        # Wellington Anniversary Day in 2025.
        "20250120|Wellington": {
            "HolidayName": "Wellington Anniversary Day",
            "IsPublicHoliday": 1,
        },
    }

    indexed_holidays = holidays_df.set_index(
        "DateRegionKey",
        drop=False,
    )

    errors: List[str] = []

    for (
        date_region_key,
        expected,
    ) in expected_results.items():
        if date_region_key not in indexed_holidays.index:
            errors.append(
                f"{date_region_key}: key not found"
            )

            continue

        row = indexed_holidays.loc[
            date_region_key
        ]

        # Defensive handling in case the key has somehow become duplicated.
        if isinstance(
            row,
            pd.DataFrame,
        ):
            errors.append(
                f"{date_region_key}: duplicate rows found"
            )

            continue

        actual_holiday_name = row[
            "HolidayName"
        ]

        actual_is_public_holiday = int(
            row["IsPublicHoliday"]
        )

        expected_holiday_name = expected[
            "HolidayName"
        ]

        expected_is_public_holiday = int(
            expected["IsPublicHoliday"]
        )

        if expected_holiday_name is None:
            holiday_name_matches = pd.isna(
                actual_holiday_name
            )
        else:
            holiday_name_matches = (
                str(actual_holiday_name)
                == expected_holiday_name
            )

        if not holiday_name_matches:
            errors.append(
                f"{date_region_key}: expected HolidayName "
                f"{expected_holiday_name!r}, got "
                f"{actual_holiday_name!r}"
            )

        if (
            actual_is_public_holiday
            != expected_is_public_holiday
        ):
            errors.append(
                f"{date_region_key}: expected IsPublicHoliday "
                f"{expected_is_public_holiday}, got "
                f"{actual_is_public_holiday}"
            )

    if errors:
        error_text = "\n".join(
            f"- {error}"
            for error in errors
        )

        raise ValueError(
            "Known holiday-date validation failed:\n"
            f"{error_text}"
        )

    print(
        "Known holiday-date examples validated successfully."
    )

 
def print_join_coverage(
    tickets_df: pd.DataFrame,
    holidays_df: pd.DataFrame,
) -> Dict[str, Any]:
    """
    Report how many ticket records have a matching holiday-calendar key.
    """

    holiday_keys = set(
        holidays_df[
            "DateRegionKey"
        ]
        .dropna()
        .astype(str)
    )

    ticket_keys = (
        tickets_df[
            "DateRegionKey"
        ]
        .astype("string")
    )

    valid_ticket_keys = (
        ticket_keys.notna()
    )

    matched_ticket_keys = (
        ticket_keys.isin(
            holiday_keys
        )
        & valid_ticket_keys
    )

    valid_count = int(
        valid_ticket_keys.sum()
    )

    matched_count = int(
        matched_ticket_keys.sum()
    )

    unmatched_count = (
        valid_count
        - matched_count
    )

    coverage = (
        matched_count / valid_count
        if valid_count
        else 0.0
    )

    print(
        f"Holiday join coverage: "
        f"{matched_count:,} of "
        f"{valid_count:,} valid ticket keys "
        f"({coverage:.1%})"
    )

    if unmatched_count:
        print(
            f"Unmatched valid ticket keys: "
            f"{unmatched_count:,}"
        )

    return {
        "valid_ticket_keys": valid_count,
        "matched_ticket_keys": matched_count,
        "unmatched_ticket_keys": unmatched_count,
        "coverage_rate": coverage,
    }


# =============================================================================
# 9. MAIN EXECUTION
# =============================================================================

def main():
    """
    Create the holiday and prepared-ticket dataframes.
    """

    OUTPUT_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True,
    )

    print()
    print("=" * 72)
    print("CREATING NEW ZEALAND HOLIDAY DATAFRAME")
    print("=" * 72)

    holidays_df = create_holiday_dataframe(
        start_date=START_DATE,
        end_date=END_DATE,
    )

    validate_holiday_dataframe(
        holidays_df
    )

    holiday_output = (
        OUTPUT_DIRECTORY
        / "nz_holiday_calendar_2023_2025.csv"
    )

    holidays_df.to_csv(
        holiday_output,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d",
    )

    print()
    print(
        f"Holiday dataframe created: "
        f"{len(holidays_df):,} rows"
    )

    print(
        f"Unique DateRegionKey values: "
        f"{holidays_df['DateRegionKey'].nunique():,}"
    )

    print(
        f"Public-holiday rows: "
        f"{holidays_df['IsPublicHoliday'].sum():,}"
    )

    print(
        f"Saved: {holiday_output}"
    )

    tickets_df: Optional[
        pd.DataFrame
    ] = None

    ticket_output: Optional[
        Path
    ] = None

    join_coverage: Optional[
        Dict[str, Any]
    ] = None

    print()
    print("=" * 72)
    print("PREPARING THE TECHSOLVE TICKET DATAFRAME")
    print("=" * 72)

    if TICKET_FILE.exists():
        tickets_df = prepare_ticket_dataframe(
            source_file=TICKET_FILE,
            sheet_name=TICKET_SHEET,
        )

        ticket_output = (
            OUTPUT_DIRECTORY
            / (
                "techsolve_tickets_with_"
                "date_region_key.csv"
            )
        )

        tickets_df.to_csv(
            ticket_output,
            index=False,
            encoding="utf-8-sig",
            date_format="%Y-%m-%d",
        )

        print(
            f"Ticket dataframe created: "
            f"{len(tickets_df):,} rows"
        )

        print(
            f"Tickets with DateRegionKey: "
            f"{tickets_df['DateRegionKey'].notna().sum():,}"
        )

        print(
            f"Saved: {ticket_output}"
        )

        join_coverage = (
            print_join_coverage(
                tickets_df=tickets_df,
                holidays_df=holidays_df,
            )
        )

    else:
        print(
            f"Ticket workbook not found: "
            f"{TICKET_FILE}"
        )

        print(
            "The holiday dataframe was still "
            "created successfully."
        )

    manifest = {
        "period": {
            "start": (
                START_DATE.isoformat()
            ),
            "end": (
                END_DATE.isoformat()
            ),
        },
        "analytical_key": (
            "DateRegionKey = "
            "YYYYMMDD|Region"
        ),
        "holiday_rows": int(
            len(holidays_df)
        ),
        "holiday_unique_keys": int(
            holidays_df[
                "DateRegionKey"
            ].nunique()
        ),
        "public_holiday_rows": int(
            holidays_df[
                "IsPublicHoliday"
            ].sum()
        ),
        "ticket_rows": (
            int(
                len(tickets_df)
            )
            if tickets_df is not None
            else None
        ),
        "join_coverage": (
            join_coverage
        ),
        "outputs": {
            "holiday_calendar": str(
                holiday_output
            ),
            "prepared_tickets": (
                str(ticket_output)
                if ticket_output is not None
                else None
            ),
        },
        "recommended_power_bi_relationship": (
            "HolidayCalendar[DateRegionKey] "
            "1 -> * "
            "FactTickets[DateRegionKey]"
        ),
        "notes": [
            (
                "National holidays were downloaded "
                "from the Nager.Date public API."
            ),
            (
                "National holidays were expanded to "
                "all TechSolve ticket regions."
            ),
            (
                "Regional anniversary dates were "
                "calculated for analytical use."
            ),
            (
                "Regional anniversary dates should "
                "be checked before payroll or legal use."
            ),
        ],
    }

    manifest_output = (
        OUTPUT_DIRECTORY
        / "download_manifest.json"
    )

    manifest_output.write_text(
        json.dumps(
            manifest,
            indent=2,
        ),
        encoding="utf-8",
    )

    print()
    print("=" * 72)
    print("COMPLETED")
    print("=" * 72)

    print(
        f"Manifest saved: "
        f"{manifest_output}"
    )

    print()
    print("Available dataframes:")

    print(
        f"holidays_df: "
        f"{holidays_df.shape}"
    )

    if tickets_df is not None:
        print(
            f"tickets_df: "
            f"{tickets_df.shape}"
        )

    return (
        holidays_df,
        tickets_df,
    )


# =============================================================================
# 10. RUN IN GOOGLE COLAB OR JUPYTER
# =============================================================================

try:
    (
        holidays_df,
        tickets_df,
    ) = main()

except KeyboardInterrupt:
    print(
        "\nProcess cancelled by the user."
    )

except Exception as exc:
    print()
    print("=" * 72)
    print("PROCESS STOPPED")
    print("=" * 72)

    print(
        f"{type(exc).__name__}: {exc}"
    )

    raise

Output folder: /content/drive/MyDrive/TechSolve_Ticket_Data
Ticket workbook: /content/drive/MyDrive/TechSolve_Ticket_Data/TechSolve - Ticket Data.xlsx
Ticket workbook exists: True

CREATING NEW ZEALAND HOLIDAY DATAFRAME

Nationwide holiday event-region rows created: 396
Regional or non-global API records excluded: 36
Excluded API holiday names:
  - Auckland/Northland Anniversary Day
  - Canterbury (North & Central) Anniversary Day
  - Chatham Islands Anniversary Day
  - Dominion Day
  - Hawke's Bay Anniversary Day
  - Marlborough Anniversary Day
  - Nelson Anniversary Day
  - Otago Anniversary Day
  - Southland Anniversary Day
  - Taranaki Anniversary Day
  - Wellington Anniversary Day
  - Westland Anniversary Day

Holiday dataframe created: 13,152 rows
Unique DateRegionKey values: 13,152
Public-holiday rows: 432
Saved: /content/drive/MyDrive/TechSolve_Ticket_Data/nz_holiday_calendar_2023_2025.csv

PREPARING THE TECHSOLVE TICKET DATAFRAME
Ticket dataframe created: 100,851 rows
Tickets 

# Data Validation Summary and Cleasing

Goal: Create one complete phython file to download in markdown style to perform a comprehensive Data Validation and Data Cleansing assessment on tickets_df before any reporting, modelling or Power BI development.

Requirements:

1. Validate every original 36 columns in tickets_df and produce a detailed Data Quality Report.

2. For each field, analyse and report:
   - Total row count
   - Null values
   - Blank values
   - NA values
   - Duplicate values
   - Invalid values
   - Outliers
   - Unexpected categories
   - Data type issues
   - Cross-field consistency issues

3. For each field:
   - If no issues are found, print:

     Field: <field_name>
     Status: ✅ VALIDATED
     Result: No issues found.

   - If issues are found, print:

     Field: <field_name>
     Status: ❌ ISSUE FOUND

     Issue:
     <issue description>

     Affected Rows:
     <row count>

     Sample Records:
     <examples>

     Recommended Action:
     <recommended remediation>

4. Mandatory validation rules:

   ticket_id
   - Must not be null
   - Must not be blank
   - Must be unique
   - Clearly report whether duplicate ticket IDs exist

   customer_id
   - Validate customer_id against customer_name
   - One customer_id should correspond to only one customer_name
   - Identify customer_ids linked to multiple customer names
   - Report affected customer_ids, row counts and examples
   - Recommend remediation

   customer_name
   - Check nulls, blanks, formatting inconsistencies, casing differences and duplicate variations

   customer_email
   - Validate email format
   - Check one email mapped to multiple customer names
   - Check one customer mapped to multiple emails

   ticket_created_date
   - Check invalid dates
   - Check future dates
   - Check outlier dates
   - Check missing dates

   ticket_resolved_date
   - Check missing dates
   - Check future dates
   - Check dates earlier than ticket_created_date

   status
   - Check missing 

   priority
   - Check missing 

   region
   - Identify unknown, blank or inconsistent values

   sla_breached
   - Compare with resolution_time_hours and sla_target_hours
   - if resolution_time_hours > sla_target_hours and sla_breached is No, then
   - Report mismatch counts

   resolution_time_hours
   - Validate numeric values
   - Check negative values

   first_response_time_hours
   - Validate numeric values
   - Check negative values

   csat_score
   - Validate values are between 1 and 5
   - Report number of out of the 1-5 values and empty

5. Perform cross-field validation:
   - customer_id → customer_name consistency
   - customer_email → customer_name consistency
   - ticket_created_date ≤ ticket_resolved_date
   - count row (Pending Customer, Open, In Progress) status but ticket_resolved_date passed

6. Produce a final Data Quality Scorecard summarising:
   - Field Name
   - Validation Status
   - Issue Description
   - Affected Rows
   - Severity (Critical, High, Medium, Low)
   - Recommended Action

7. Produce an Executive Summary including:
   - Total row count
   - Number of validated fields
   - Number of fields with issues
   - Total critical issues
   - Top data quality risks
   - Recommended cleansing actions before reporting

Do not modify data. Perform assessment only and provide findings, evidence, row counts, examples and recommended remediation actions.


This script performs an assessment only on the existing tickets_df. It does not modify, delete, replace or deduplicate any source values.
It:

- validates all 36 original fields;
- applies generic and field-specific checks;
- performs mandatory cross-field validation;
- prints a clear result for every field;
- creates `data_quality_scorecard_df`;
- creates `data_quality_issues_df`;
- creates executive_summary;
- saves the assessment outputs to Google Drive.

## step 1 mount Google Drive and add the folder to Python’s path

In [4]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount(
    "/content/drive"
)

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/"
    "TechSolve_Ticket_Data"
)

MODULE_FILE = (
    PROJECT_DIRECTORY
    / "validate_techsolve_tickets.py"
)

print(
    f"Project folder exists: "
    f"{PROJECT_DIRECTORY.exists()}"
)

print(
    f"Validation script exists: "
    f"{MODULE_FILE.exists()}"
)

if not MODULE_FILE.exists():
    available_files = [
        path.name
        for path in PROJECT_DIRECTORY.iterdir()
        if path.is_file()
    ]

    raise FileNotFoundError(
        f"Python validation file was not found:\n"
        f"{MODULE_FILE}\n\n"
        f"Files currently in the folder:\n"
        f"{available_files}"
    )

project_path_text = str(
    PROJECT_DIRECTORY
)

if project_path_text not in sys.path:
    sys.path.insert(
        0,
        project_path_text,
    )

print(
    f"Added to Python import path: "
    f"{project_path_text}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder exists: True
Validation script exists: True
Added to Python import path: /content/drive/MyDrive/TechSolve_Ticket_Data


## step 2 import and run the validation

In [5]:
# validate_techsolve_tickets.py in google colab
from validate_techsolve_tickets import (
    run_data_quality_assessment,
)

results = run_data_quality_assessment(
    tickets_df=tickets_df,
    output_dir=(
        "/content/drive/MyDrive/"
        "TechSolve_Ticket_Data/"
        "data_quality_reports"
    ),
    expected_start_date="2023-01-01",
    expected_end_date="2025-12-31",
    sample_size=5,
    print_report=True,
)

TECHSOLVE DATA VALIDATION AND CLEANSING ASSESSMENT

Field: ticket_id
Status: ✅ VALIDATED
Result: No issues found.

Field Profile:
- Total Rows: 100,851
- Null Values: 0
- Blank Values: 0
- NA Text Values: 0
- Unique Non-null Values: 100,851
- Duplicate Rows (all occurrences): 0
- Duplicate Rows (beyond first): 0
- Invalid Values: 0
- Outlier Values: 0
- Unexpected Categories: 0

Field: customer_id
Status: ❌ ISSUE FOUND

Issue 1:
One customer_id is associated with multiple customer names.

Affected Rows:
98,408

Sample Records:
[
  {
    "customer_id": "ACC-00002",
    "distinct_customer_names": 24,
    "customer_name_examples": [
      "Charles Martinez",
      "Elizabeth Smith",
      "James Moore",
      "James Rodriguez",
      "Jennifer Moore",
      "Jessica Davis",
      "Jessica Martin",
      "Jessica Miller",
      "John Gonzalez",
      "John Miller"
    ]
  },
  {
    "customer_id": "ACC-00008",
    "distinct_customer_names": 8,
    "customer_name_examples": [
      "David M

## step 3 — access the results

In [6]:
field_profile_df = results[
    "field_profile_df"
]

data_quality_issues_df = results[
    "data_quality_issues_df"
]

data_quality_scorecard_df = results[
    "data_quality_scorecard_df"
]

executive_summary = results[
    "executive_summary"
]

# Data Validation Report

now, according the results, Create one complete phython file to download in markdown style, new dataframe from original tickets_df dataframe by add new fields for filter these results. Naming the new field with name related to the original field and prefix is "RK_".  

The script creates a new dataframe named:

 rk_tickets_df Show more lines

It preserves every value and column in the original tickets_df, then adds filter-ready fields beginning with:

RK_ Show more lines

The script was syntax-checked and successfully tested against the TechSolve workbook containing 100,851 rows and 36 original fields. The test retained all 100,851 rows and did not modify the source dataframe.

*Decision supported*

The new dataframe helps analysts decide:

- which records are suitable for a particular report;
- which records require data-quality review;
- which specific source field caused an issue;
- whether the issue is Low, Medium, High or Critical;
- whether reported SLA results disagree with the provisional calculation.


RK_Row_ReportingEligible is a provisional filter, not an automatic deletion rule. In particular, the widespread customer identifier conflict should be resolved through a customer master rather than by removing nearly all affected tickets.

In [7]:
from pathlib import Path
import sys

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/"
    "TechSolve_Ticket_Data"
)

OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "rk_quality_outputs"
)

OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

# Allow Python to import the custom script from Google Drive.
if str(PROJECT_DIRECTORY) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_DIRECTORY),
    )

from create_rk_ticket_quality_flags import (
    create_rk_ticket_quality_flags,
)

# Create the new dataframes.
(
    rk_tickets_df,
    rk_flag_dictionary_df,
    rk_quality_summary_df,
) = create_rk_ticket_quality_flags(
    tickets_df=tickets_df,
    output_dir=str(
        OUTPUT_DIRECTORY
    ),
    expected_start_date="2023-01-01",
    expected_end_date="2025-12-31",
)

print()
print("=" * 72)
print("DATAFRAMES SAVED")
print("=" * 72)

print(
    f"rk_tickets_df: "
    f"{rk_tickets_df.shape}"
)

print(
    f"rk_flag_dictionary_df: "
    f"{rk_flag_dictionary_df.shape}"
)

print(
    f"rk_quality_summary_df: "
    f"{rk_quality_summary_df.shape}"
)

print()
print(
    f"Output directory: "
    f"{OUTPUT_DIRECTORY}"
)

/content/drive/MyDrive/TechSolve_Ticket_Data/create_rk_ticket_quality_flags.py:195: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  rk_tickets_df[name] = _flag(aligned)
/content/drive/MyDrive/TechSolve_Ticket_Data/create_rk_ticket_quality_flags.py:195: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  rk_tickets_df[name] = _flag(aligned)
/content/drive/MyDrive/TechSolve_Ticket_Data/create_rk_ticket_quality_flags.py:195: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times

RK QUALITY-FLAG DATAFRAME CREATED
Source rows: 100,851
Output rows: 100,851
Source columns: 40
RK_ fields added: 357
Rows with any issue: 100,843
Provisionally reporting-eligible rows: 460
Original tickets_df was not modified.

DATAFRAMES SAVED
rk_tickets_df: (100851, 397)
rk_flag_dictionary_df: (357, 5)
rk_quality_summary_df: (318, 3)

Output directory: /content/drive/MyDrive/TechSolve_Ticket_Data/rk_quality_outputs


# Marge holidays_df into tickets_df
combine tickets_df and holidays_df with the key field: DateKey and DateRegionKey to new dataframe. 

In [8]:
import pandas as pd


# =============================================================================
# 1. CREATE COPIES
# =============================================================================
# The original tickets_df and holidays_df are not modified.

tickets_merge_df = tickets_df.copy(
    deep=True
)

holidays_merge_df = holidays_df.copy(
    deep=True
)


# =============================================================================
# 2. CHECK REQUIRED FIELDS
# =============================================================================

required_ticket_fields = {
    "ticket_created_date",
    "region",
}

missing_ticket_fields = (
    required_ticket_fields
    .difference(
        tickets_merge_df.columns
    )
)

if missing_ticket_fields:
    raise ValueError(
        "tickets_df is missing required fields: "
        f"{sorted(missing_ticket_fields)}"
    )


required_holiday_fields = {
    "Date",
    "Region",
}

missing_holiday_fields = (
    required_holiday_fields
    .difference(
        holidays_merge_df.columns
    )
)

if missing_holiday_fields:
    raise ValueError(
        "holidays_df is missing required fields: "
        f"{sorted(missing_holiday_fields)}"
    )


# =============================================================================
# 3. CREATE/CLEAN TICKET KEYS
# =============================================================================

# Create TicketDate if it does not already exist.
if "TicketDate" not in tickets_merge_df.columns:
    tickets_merge_df[
        "TicketDate"
    ] = pd.to_datetime(
        tickets_merge_df[
            "ticket_created_date"
        ],
        errors="coerce",
    ).dt.normalize()

else:
    tickets_merge_df[
        "TicketDate"
    ] = pd.to_datetime(
        tickets_merge_df[
            "TicketDate"
        ],
        errors="coerce",
    ).dt.normalize()


# Create RegionAnalytics if it does not already exist.
if "RegionAnalytics" not in tickets_merge_df.columns:
    tickets_merge_df[
        "RegionAnalytics"
    ] = (
        tickets_merge_df[
            "region"
        ]
        .astype("string")
        .str.replace(
            "\r",
            "",
            regex=False,
        )
        .str.replace(
            "\n",
            "",
            regex=False,
        )
        .str.strip()
    )

else:
    tickets_merge_df[
        "RegionAnalytics"
    ] = (
        tickets_merge_df[
            "RegionAnalytics"
        ]
        .astype("string")
        .str.replace(
            "\r",
            "",
            regex=False,
        )
        .str.replace(
            "\n",
            "",
            regex=False,
        )
        .str.strip()
    )


# Create DateKey as YYYYMMDD.
tickets_merge_df[
    "DateKey"
] = (
    tickets_merge_df[
        "TicketDate"
    ]
    .dt.strftime(
        "%Y%m%d"
    )
    .astype("string")
)


# Create DateRegionKey.
tickets_merge_df[
    "DateRegionKey"
] = (
    tickets_merge_df[
        "DateKey"
    ]
    + "|"
    + tickets_merge_df[
        "RegionAnalytics"
    ]
)


# Prevent invalid keys when date or region is missing.
invalid_ticket_key = (
    tickets_merge_df[
        "TicketDate"
    ].isna()
    | tickets_merge_df[
        "RegionAnalytics"
    ].isna()
    | tickets_merge_df[
        "RegionAnalytics"
    ].str.strip().eq("")
)

tickets_merge_df.loc[
    invalid_ticket_key,
    "DateRegionKey",
] = pd.NA


# Clean hidden newline characters.
tickets_merge_df[
    "DateRegionKey"
] = (
    tickets_merge_df[
        "DateRegionKey"
    ]
    .astype("string")
    .str.replace(
        "\r",
        "",
        regex=False,
    )
    .str.replace(
        "\n",
        "",
        regex=False,
    )
    .str.strip()
)


# =============================================================================
# 4. CREATE/CLEAN HOLIDAY KEYS
# =============================================================================

holidays_merge_df[
    "Date"
] = pd.to_datetime(
    holidays_merge_df[
        "Date"
    ],
    errors="coerce",
).dt.normalize()


holidays_merge_df[
    "Region"
] = (
    holidays_merge_df[
        "Region"
    ]
    .astype("string")
    .str.replace(
        "\r",
        "",
        regex=False,
    )
    .str.replace(
        "\n",
        "",
        regex=False,
    )
    .str.strip()
)


# Create DateKey if it does not exist.
holidays_merge_df[
    "DateKey"
] = (
    holidays_merge_df[
        "Date"
    ]
    .dt.strftime(
        "%Y%m%d"
    )
    .astype("string")
)


# Recreate DateRegionKey from the cleaned date and region.
holidays_merge_df[
    "DateRegionKey"
] = (
    holidays_merge_df[
        "DateKey"
    ]
    + "|"
    + holidays_merge_df[
        "Region"
    ]
)


invalid_holiday_key = (
    holidays_merge_df[
        "Date"
    ].isna()
    | holidays_merge_df[
        "Region"
    ].isna()
    | holidays_merge_df[
        "Region"
    ].str.strip().eq("")
)

holidays_merge_df.loc[
    invalid_holiday_key,
    "DateRegionKey",
] = pd.NA


holidays_merge_df[
    "DateRegionKey"
] = (
    holidays_merge_df[
        "DateRegionKey"
    ]
    .astype("string")
    .str.replace(
        "\r",
        "",
        regex=False,
    )
    .str.replace(
        "\n",
        "",
        regex=False,
    )
    .str.strip()
)


# =============================================================================
# 5. VALIDATE HOLIDAY KEY UNIQUENESS
# =============================================================================

duplicate_holiday_keys = (
    holidays_merge_df[
        "DateRegionKey"
    ]
    .notna()
    & holidays_merge_df[
        "DateRegionKey"
    ]
    .duplicated(
        keep=False
    )
)


if duplicate_holiday_keys.any():
    duplicate_examples = (
        holidays_merge_df.loc[
            duplicate_holiday_keys,
            [
                "DateKey",
                "DateRegionKey",
                "Date",
                "Region",
                "HolidayName",
            ],
        ]
        .sort_values(
            [
                "DateRegionKey",
                "HolidayName",
            ]
        )
        .head(20)
    )

    print(
        "Duplicate holiday keys:"
    )

    display(
        duplicate_examples
    )

    raise ValueError(
        "holidays_df contains duplicate "
        "DateRegionKey values. The join would "
        "duplicate ticket records."
    )


# Validate the combination of both requested keys.
duplicate_holiday_combined_keys = (
    holidays_merge_df[
        [
            "DateKey",
            "DateRegionKey",
        ]
    ]
    .duplicated(
        keep=False
    )
    & holidays_merge_df[
        "DateRegionKey"
    ].notna()
)


if duplicate_holiday_combined_keys.any():
    raise ValueError(
        "holidays_df contains duplicate combinations "
        "of DateKey and DateRegionKey."
    )


# =============================================================================
# 6. SELECT HOLIDAY FIELDS
# =============================================================================

holiday_fields_to_merge = [
    "DateKey",
    "DateRegionKey",
    "Date",
    "Region",
    "Year",
    "MonthNumber",
    "MonthName",
    "YearMonth",
    "DayName",
    "HolidayName",
    "HolidayScope",
    "HolidaySource",
    "IsPublicHoliday",
    "FollowingHolidayName",
    "FollowingHolidayScope",
    "IsDayBeforeHoliday",
    "PreviousHolidayName",
    "PreviousHolidayScope",
    "IsDayAfterHoliday",
    "IsWeekend",
    "IsBusinessDay",
]


# Keep only fields that actually exist.
holiday_fields_to_merge = [
    field
    for field in holiday_fields_to_merge
    if field in holidays_merge_df.columns
]


holiday_dimension_df = (
    holidays_merge_df[
        holiday_fields_to_merge
    ]
    .copy()
)


# Rename fields that might be confused with ticket fields.
holiday_dimension_df = (
    holiday_dimension_df.rename(
        columns={
            "Date": "HolidayCalendarDate",
            "Region": "HolidayRegion",
            "Year": "HolidayYear",
            "MonthNumber": (
                "HolidayMonthNumber"
            ),
            "MonthName": (
                "HolidayMonthName"
            ),
            "YearMonth": (
                "HolidayYearMonth"
            ),
            "DayName": (
                "HolidayDayName"
            ),
        }
    )
)


# =============================================================================
# 7. MERGE USING DateKey AND DateRegionKey
# =============================================================================

ticket_rows_before_merge = len(
    tickets_merge_df
)


tickets_holidays_df = (
    tickets_merge_df.merge(
        holiday_dimension_df,

        on=[
            "DateKey",
            "DateRegionKey",
        ],

        how="left",

        # Multiple tickets can match one holiday-calendar row.
        validate="many_to_one",

        # Identify successful and unsuccessful joins.
        indicator="RK_HolidayMergeStatus",
    )
)


ticket_rows_after_merge = len(
    tickets_holidays_df
)


# =============================================================================
# 8. CREATE JOIN-QUALITY FILTERS
# =============================================================================

tickets_holidays_df[
    "RK_HolidayMatched"
] = (
    tickets_holidays_df[
        "RK_HolidayMergeStatus"
    ]
    .eq("both")
    .astype("int8")
)


tickets_holidays_df[
    "RK_HolidayUnmatched"
] = (
    tickets_holidays_df[
        "RK_HolidayMergeStatus"
    ]
    .eq("left_only")
    .astype("int8")
)


tickets_holidays_df[
    "RK_HolidayMergeStatus"
] = (
    tickets_holidays_df[
        "RK_HolidayMergeStatus"
    ]
    .astype("string")
    .replace(
        {
            "both": "Matched",
            "left_only": (
                "Ticket Key Not Found "
                "in Holiday Calendar"
            ),
            "right_only": (
                "Holiday Without Ticket"
            ),
        }
    )
)


# Do not convert missing holiday flags to zero automatically.
#
# A missing value can mean the ticket did not match the calendar,
# which is different from a matched date that is not a holiday.


# =============================================================================
# 9. VALIDATE MERGE RESULT
# =============================================================================

if (
    ticket_rows_after_merge
    != ticket_rows_before_merge
):
    raise ValueError(
        "The merge changed the ticket row count. "
        f"Before: {ticket_rows_before_merge:,}; "
        f"after: {ticket_rows_after_merge:,}."
    )


matched_count = int(
    tickets_holidays_df[
        "RK_HolidayMatched"
    ].sum()
)


unmatched_count = int(
    tickets_holidays_df[
        "RK_HolidayUnmatched"
    ].sum()
)


valid_ticket_key_count = int(
    tickets_holidays_df[
        "DateRegionKey"
    ].notna().sum()
)


coverage_rate = (
    matched_count
    / valid_ticket_key_count
    if valid_ticket_key_count
    else 0.0
)


print("=" * 72)
print("TICKET AND HOLIDAY MERGE COMPLETED")
print("=" * 72)

print(
    f"Ticket rows before merge: "
    f"{ticket_rows_before_merge:,}"
)

print(
    f"Rows after merge: "
    f"{ticket_rows_after_merge:,}"
)

print(
    f"Valid ticket keys: "
    f"{valid_ticket_key_count:,}"
)

print(
    f"Matched rows: "
    f"{matched_count:,}"
)

print(
    f"Unmatched rows: "
    f"{unmatched_count:,}"
)

print(
    f"Holiday join coverage: "
    f"{coverage_rate:.1%}"
)

print(
    f"New dataframe: "
    f"tickets_holidays_df "
    f"{tickets_holidays_df.shape}"
)

from pathlib import Path

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/"
    "TechSolve_Ticket_Data"
)

OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "rk_quality_outputs"
)

OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


TICKETS_HOLIDAYS_OUTPUT = (
    OUTPUT_DIRECTORY
    / "tickets_holidays_df.csv"
)


tickets_holidays_df.to_csv(
    TICKETS_HOLIDAYS_OUTPUT,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)


print(
    f"Saved: "
    f"{TICKETS_HOLIDAYS_OUTPUT}"
)

print(
    f"File exists: "
    f"{TICKETS_HOLIDAYS_OUTPUT.exists()}"
)

print(
    f"File size: "
    f"{TICKETS_HOLIDAYS_OUTPUT.stat().st_size:,} "
    f"bytes"
)

# out: tickets_holidays_df

TICKET AND HOLIDAY MERGE COMPLETED
Ticket rows before merge: 100,851
Rows after merge: 100,851
Valid ticket keys: 100,851
Matched rows: 100,843
Unmatched rows: 8
Holiday join coverage: 100.0%
New dataframe: tickets_holidays_df (100851, 62)
Saved: /content/drive/MyDrive/TechSolve_Ticket_Data/rk_quality_outputs/tickets_holidays_df.csv
File exists: True
File size: 58,429,873 bytes


In [9]:
tickets_holidays_df

,ticket_id,customer_id,customer_name,customer_email,company_name,account_type,customer_segment,industry,billing_contact_email,account_manager,...,FollowingHolidayScope,IsDayBeforeHoliday,PreviousHolidayName,PreviousHolidayScope,IsDayAfterHoliday,IsWeekend,IsBusinessDay,RK_HolidayMergeStatus,RK_HolidayMatched,RK_HolidayUnmatched
0,1,ACC-07512,David Martin,david.martin4@hotmail.com,Central Ltd,Business,Small Business,Retail,NaN,Sarah Chen,...,NaN,0.0,NaN,NaN,0.0,1.0,0.0,Matched,1,0
1,2,ACC-07455,Patricia Martin,patricia.martin869@icloud.com,Gateway Transport,Business,Small Business,Trades,NaN,NaN,...,NaN,0.0,NaN,NaN,0.0,1.0,0.0,Matched,1,0
2,3,ACC-03650,John Lopez,john.lopez551@gmail.com,Pioneer Logistics,Business,Small Business,Trades,NaN,NaN,...,NaN,0.0,NaN,NaN,0.0,1.0,0.0,Matched,1,0
3,4,ACC-05579,Mary Martinez,mary.martinez515@company.com,Coastal Tech,Business,Small Business,Technology,NaN,NaN,...,NaN,0.0,NaN,NaN,0.0,1.0,0.0,Matched,1,0
4,5,ACC-07456,John Martinez,john.martinez733@hotmail.com,NaN,Residential,Individual,NaN,NaN,NaN,...,NaN,0.0,NaN,NaN,0.0,1.0,0.0,Matched,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100846,100847,ACC-06526,Charles Johnson,charles.johnson457@yahoo.com,NaN,Residential,Individual,NaN,NaN,NaN,...,National,1.0,NaN,NaN,0.0,0.0,1.0,Matched,1,0
100847,100848,ACC-03650,William Brown,william.brown671@hotmail.com,Pioneer Logistics,Business,Small Business,Trades,NaN,NaN,...,National,1.0,NaN,NaN,0.0,0.0,1.0,Matched,1,0
100848,100849,ACC-01465,Charles Taylor,charles.taylor867@gmail.com,NaN,Residential,Individual,NaN,NaN,NaN,...,National,1.0,NaN,NaN,0.0,0.0,1.0,Matched,1,0
100849,100850,ACC-03432,Susan Thomas,susan.thomas126@gmail.com,NaN,Residential,Small Business,NaN,NaN,Rob Thompson,...,National,1.0,NaN,NaN,0.0,0.0,1.0,Matched,1,0


# Create Categories for Operation manager

support streamlining support ticket issues into a consistent set of categories and sub-categories suitable for 
operations manager has asked for better visibility into customer support performance. They want to understand what types of issues are being raised, how they're being handled, and where there's room to improve.

okay, using tickets_holidays_df, add suggestion
6 categories,
14 sub-categories and 
a clear split between operational incidents, commercial requests, enhancement demand, security concerns and data-quality exceptions.

| Field                       | Purpose                                                  |
| --------------------------- | -------------------------------------------------------- |
| `RK_Category`               | Broad operational issue group                            |

1. Account & Access
2. Billing, Payments & Subscription
3. Product Reliability & Defects
4. Product Improvement
5. Security & Privacy
6. Unclassified & Data Quality

| Field                       | Purpose                                                  |
| --------------------------- | -------------------------------------------------------- |
| `RK_SubCategory`            | Specific support reason                                  |

1. Login Issue
2. Account Suspension
3. Payment Problem
4. Refund Request
5. Subscription Cancellation
6. Bug Report
7. Performance Issue
8. Data Sync Issue
9. Feature Request
10. Security Concern
11. Missing Category
12. Unmapped Category
13. Ambiguous Classification
14. Manual Review Required

| Field                       | Purpose                                                  |
| --------------------------- | -------------------------------------------------------- |
| `RK_CategoryMappingStatus`  | Mapped, Missing, Unmapped or Ambiguous                   |
| `RK_CategoryMappingMethod`  | Exact, Normalised Alias, Description Rule or Manual      |
| `RK_CategoryReviewRequired` | `1` when classification needs review                     |
| `RK_OperationalOwner`       | Suggested operational team                               |
| `RK_IssueNature`            | Incident, Request, Security or Commercial                |
| `RK_ImprovementTheme`       | Automation, Product Fix, Process Fix, Training or Review |

Keep the original category field unchanged for auditability.

see output: tickets_holidays_categorised_df


In [10]:
#!/usr/bin/env python3
"""
Add a controlled operational support taxonomy to tickets_holidays_df.

The original category field remains unchanged.

New dataframe
-------------
tickets_holidays_categorised_df

New fields
----------
RK_Category
RK_SubCategory
RK_CategoryMappingStatus
RK_CategoryMappingMethod
RK_CategoryReviewRequired
RK_OperationalOwner
RK_IssueNature
RK_ImprovementTheme
RK_CategoryNormalised

Additional outputs
------------------
rk_taxonomy_dictionary_df
rk_category_mapping_summary_df
rk_unmapped_categories_df
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Set

import pandas as pd


# =============================================================================
# 1. VALIDATE INPUT DATAFRAME
# =============================================================================

if "tickets_holidays_df" not in globals():
    raise NameError(
        "tickets_holidays_df was not found. "
        "Create the combined ticket and holiday dataframe first."
    )

if not isinstance(
    tickets_holidays_df,
    pd.DataFrame,
):
    raise TypeError(
        "tickets_holidays_df exists but is not a pandas DataFrame."
    )

required_fields = {
    "category",
}

missing_fields = (
    required_fields
    .difference(
        tickets_holidays_df.columns
    )
)

if missing_fields:
    raise ValueError(
        "tickets_holidays_df is missing required fields: "
        f"{sorted(missing_fields)}"
    )


# =============================================================================
# 2. PRESERVE THE ORIGINAL DATAFRAME
# =============================================================================

tickets_holidays_categorised_df = (
    tickets_holidays_df.copy(
        deep=True
    )
)

source_row_count = len(
    tickets_holidays_df
)


# Confirm that the original category field is available.
original_category_snapshot = (
    tickets_holidays_df[
        "category"
    ].copy(
        deep=True
    )
)


# =============================================================================
# 3. TEXT NORMALISATION
# =============================================================================

def normalise_issue_text(
    value: Any,
) -> Optional:
    """
    Convert source text into a comparison token.

    The function is used only for matching. It does not change the original
    category or description fields.

    Examples
    --------
    'Refund_Request' -> 'refund request'
    '  LOGIN ISSUE ' -> 'login issue'
    'Bug Report!'    -> 'bug report'
    """

    if pd.isna(value):
        return None

    text = str(value)

    text = (
        text
        .replace("\r", " ")
        .replace("\n", " ")
        .replace("_", " ")
        .strip()
        .casefold()
    )

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text or None


tickets_holidays_categorised_df[
    "RK_CategoryNormalised"
] = (
    tickets_holidays_categorised_df[
        "category"
    ].map(
        normalise_issue_text
    )
)


# =============================================================================
# 4. CONTROLLED TAXONOMY
# =============================================================================

TAXONOMY: List[Dict[str, Any]] = [
    # -------------------------------------------------------------------------
    # Account & Access
    # -------------------------------------------------------------------------
    {
        "RK_Category": "Account & Access",
        "RK_SubCategory": "Login Issue",
        "RK_OperationalOwner": (
            "Support / Identity"
        ),
        "RK_IssueNature": "Incident",
        "RK_ImprovementTheme": "Automation",
        "Aliases": {
            "login issue",
            "login problem",
            "sign in issue",
            "signin issue",
            "authentication issue",
            "password issue",
            "account login",
        },
        "DescriptionKeywords": {
            "cannot login",
            "cannot log in",
            "unable to login",
            "unable to log in",
            "sign in",
            "signin",
            "password reset",
            "authentication",
            "multi factor",
            "mfa",
            "account locked",
        },
    },
    {
        "RK_Category": "Account & Access",
        "RK_SubCategory": "Account Suspension",
        "RK_OperationalOwner": (
            "Credit / Account Administration"
        ),
        "RK_IssueNature": "Incident",
        "RK_ImprovementTheme": "Process Fix",
        "Aliases": {
            "account suspension",
            "acct suspension",
            "suspended account",
            "account suspended",
        },
        "DescriptionKeywords": {
            "account suspended",
            "account blocked",
            "account disabled",
            "reactivate account",
            "account reactivation",
            "suspension",
        },
    },

    # -------------------------------------------------------------------------
    # Billing, Payments & Subscription
    # -------------------------------------------------------------------------
    {
        "RK_Category": (
            "Billing, Payments & Subscription"
        ),
        "RK_SubCategory": "Payment Problem",
        "RK_OperationalOwner": (
            "Billing / Payments"
        ),
        "RK_IssueNature": "Commercial",
        "RK_ImprovementTheme": "Process Fix",
        "Aliases": {
            "payment problem",
            "payment issue",
            "payment failure",
            "failed payment",
        },
        "DescriptionKeywords": {
            "payment failed",
            "payment declined",
            "card declined",
            "duplicate charge",
            "charged twice",
            "payment missing",
            "payment not showing",
            "transaction failed",
        },
    },
    {
        "RK_Category": (
            "Billing, Payments & Subscription"
        ),
        "RK_SubCategory": "Refund Request",
        "RK_OperationalOwner": (
            "Billing / Credit"
        ),
        "RK_IssueNature": "Commercial",
        "RK_ImprovementTheme": "Process Fix",
        "Aliases": {
            "refund request",
            "refund req",
            "refund",
        },
        "DescriptionKeywords": {
            "request refund",
            "refund status",
            "refund not received",
            "refund missing",
            "refund amount",
            "want a refund",
        },
    },
    {
        "RK_Category": (
            "Billing, Payments & Subscription"
        ),
        "RK_SubCategory": (
            "Subscription Cancellation"
        ),
        "RK_OperationalOwner": (
            "Subscription Operations"
        ),
        "RK_IssueNature": "Commercial",
        "RK_ImprovementTheme": "Automation",
        "Aliases": {
            "subscription cancellation",
            "subscription cancel",
            "sub cancellation",
            "cancel subscription",
        },
        "DescriptionKeywords": {
            "cancel subscription",
            "cancel my subscription",
            "unable to cancel",
            "subscription cancellation",
            "unwanted renewal",
            "stop renewal",
            "terminate subscription",
        },
    },

    # -------------------------------------------------------------------------
    # Product Reliability & Defects
    # -------------------------------------------------------------------------
    {
        "RK_Category": (
            "Product Reliability & Defects"
        ),
        "RK_SubCategory": "Bug Report",
        "RK_OperationalOwner": (
            "Product / Engineering"
        ),
        "RK_IssueNature": "Incident",
        "RK_ImprovementTheme": "Product Fix",
        "Aliases": {
            "bug report",
            "bug",
            "software bug",
            "defect",
        },
        "DescriptionKeywords": {
            "bug",
            "error message",
            "application error",
            "incorrect behaviour",
            "not working",
            "does not work",
            "crash",
            "system error",
            "unexpected error",
        },
    },
    {
        "RK_Category": (
            "Product Reliability & Defects"
        ),
        "RK_SubCategory": "Performance Issue",
        "RK_OperationalOwner": (
            "Platform / Engineering"
        ),
        "RK_IssueNature": "Incident",
        "RK_ImprovementTheme": "Product Fix",
        "Aliases": {
            "performance issue",
            "perf issue",
            "slow performance",
            "performance problem",
        },
        "DescriptionKeywords": {
            "slow",
            "slowness",
            "performance",
            "latency",
            "timeout",
            "timed out",
            "loading slowly",
            "slow response",
            "degraded",
        },
    },
    {
        "RK_Category": (
            "Product Reliability & Defects"
        ),
        "RK_SubCategory": "Data Sync Issue",
        "RK_OperationalOwner": (
            "Integration / Data Engineering"
        ),
        "RK_IssueNature": "Incident",
        "RK_ImprovementTheme": "Product Fix",
        "Aliases": {
            "data sync issue",
            "data sync",
            "sync issue",
            "synchronisation issue",
            "synchronization issue",
        },
        "DescriptionKeywords": {
            "data not syncing",
            "not synchronising",
            "not synchronizing",
            "sync delayed",
            "delayed sync",
            "missing records",
            "duplicate records",
            "stale data",
            "data mismatch",
        },
    },

    # -------------------------------------------------------------------------
    # Product Improvement
    # -------------------------------------------------------------------------
    {
        "RK_Category": "Product Improvement",
        "RK_SubCategory": "Feature Request",
        "RK_OperationalOwner": (
            "Product Management"
        ),
        "RK_IssueNature": "Request",
        "RK_ImprovementTheme": "Review",
        "Aliases": {
            "feature request",
            "feature req",
            "enhancement request",
            "product enhancement",
        },
        "DescriptionKeywords": {
            "new feature",
            "feature request",
            "would like",
            "please add",
            "enhancement",
            "new functionality",
            "improve feature",
        },
    },

    # -------------------------------------------------------------------------
    # Security & Privacy
    # -------------------------------------------------------------------------
    {
        "RK_Category": "Security & Privacy",
        "RK_SubCategory": "Security Concern",
        "RK_OperationalOwner": (
            "Security / Risk"
        ),
        "RK_IssueNature": "Security",
        "RK_ImprovementTheme": "Review",
        "Aliases": {
            "security concern",
            "security",
            "security issue",
            "privacy concern",
        },
        "DescriptionKeywords": {
            "unauthorised access",
            "unauthorized access",
            "suspicious login",
            "compromised",
            "phishing",
            "vulnerability",
            "security alert",
            "data exposure",
            "privacy",
            "security breach",
        },
    },

    # -------------------------------------------------------------------------
    # Unclassified & Data Quality
    # -------------------------------------------------------------------------
    {
        "RK_Category": (
            "Unclassified & Data Quality"
        ),
        "RK_SubCategory": "Missing Category",
        "RK_OperationalOwner": (
            "Support Operations"
        ),
        "RK_IssueNature": (
            "Data Quality Exception"
        ),
        "RK_ImprovementTheme": "Training",
        "Aliases": set(),
        "DescriptionKeywords": set(),
    },
    {
        "RK_Category": (
            "Unclassified & Data Quality"
        ),
        "RK_SubCategory": "Unmapped Category",
        "RK_OperationalOwner": (
            "Data Governance"
        ),
        "RK_IssueNature": (
            "Data Quality Exception"
        ),
        "RK_ImprovementTheme": "Review",
        "Aliases": set(),
        "DescriptionKeywords": set(),
    },
    {
        "RK_Category": (
            "Unclassified & Data Quality"
        ),
        "RK_SubCategory": (
            "Ambiguous Classification"
        ),
        "RK_OperationalOwner": (
            "Support Quality"
        ),
        "RK_IssueNature": (
            "Data Quality Exception"
        ),
        "RK_ImprovementTheme": "Review",
        "Aliases": set(),
        "DescriptionKeywords": set(),
    },
    {
        "RK_Category": (
            "Unclassified & Data Quality"
        ),
        "RK_SubCategory": (
            "Manual Review Required"
        ),
        "RK_OperationalOwner": (
            "Support Quality"
        ),
        "RK_IssueNature": (
            "Data Quality Exception"
        ),
        "RK_ImprovementTheme": "Review",
        "Aliases": set(),
        "DescriptionKeywords": set(),
    },
]


# =============================================================================
# 5. BUILD LOOKUP TABLES
# =============================================================================

alias_lookup: Dict[
    str,
    List[Dict[str, Any]],
] = {}


for taxonomy_item in TAXONOMY:
    for alias in taxonomy_item[
        "Aliases"
    ]:
        alias_token = normalise_issue_text(
            alias
        )

        if alias_token is None:
            continue

        alias_lookup.setdefault(
            alias_token,
            [],
        ).append(
            taxonomy_item
        )


def create_taxonomy_result(
    category: str,
    subcategory: str,
    mapping_status: str,
    mapping_method: str,
    review_required: int,
    operational_owner: str,
    issue_nature: str,
    improvement_theme: str,
) -> Dict[str, Any]:
    """
    Return a standard classification result.
    """

    return {
        "RK_Category": category,
        "RK_SubCategory": subcategory,
        "RK_CategoryMappingStatus": (
            mapping_status
        ),
        "RK_CategoryMappingMethod": (
            mapping_method
        ),
        "RK_CategoryReviewRequired": int(
            review_required
        ),
        "RK_OperationalOwner": (
            operational_owner
        ),
        "RK_IssueNature": issue_nature,
        "RK_ImprovementTheme": (
            improvement_theme
        ),
    }


def classification_from_taxonomy_item(
    taxonomy_item: Dict[str, Any],
    mapping_method: str,
) -> Dict[str, Any]:
    """
    Convert one taxonomy item into dataframe fields.
    """

    return create_taxonomy_result(
        category=taxonomy_item[
            "RK_Category"
        ],
        subcategory=taxonomy_item[
            "RK_SubCategory"
        ],
        mapping_status="Mapped",
        mapping_method=mapping_method,
        review_required=0,
        operational_owner=taxonomy_item[
            "RK_OperationalOwner"
        ],
        issue_nature=taxonomy_item[
            "RK_IssueNature"
        ],
        improvement_theme=taxonomy_item[
            "RK_ImprovementTheme"
        ],
    )


# =============================================================================
# 6. CLASSIFICATION LOGIC
# =============================================================================

def classify_ticket(
    row: pd.Series,
) -> pd.Series:
    """
    Classify one ticket.

    Mapping priority
    ----------------
    1. Missing-category rule
    2. Exact source-category alias
    3. Normalised source-category alias
    4. issue_description keyword rule
    5. Ambiguous classification
    6. Unmapped category

    The original category value is never changed.
    """

    raw_category = row.get(
        "category"
    )

    normalised_category = (
        normalise_issue_text(
            raw_category
        )
    )

    # -------------------------------------------------------------------------
    # Missing category
    # -------------------------------------------------------------------------

    if normalised_category is None:
        return pd.Series(
            create_taxonomy_result(
                category=(
                    "Unclassified & Data Quality"
                ),
                subcategory="Missing Category",
                mapping_status="Missing",
                mapping_method=(
                    "No Source Value"
                ),
                review_required=1,
                operational_owner=(
                    "Support Operations"
                ),
                issue_nature=(
                    "Data Quality Exception"
                ),
                improvement_theme="Training",
            )
        )

    # -------------------------------------------------------------------------
    # Exact source alias
    # -------------------------------------------------------------------------

    exact_matches = (
        alias_lookup.get(
            normalised_category,
            [],
        )
    )

    if len(exact_matches) == 1:
        source_text = str(
            raw_category
        ).strip()

        exact_alias_texts = {
            str(alias).strip()
            for alias in exact_matches[
                0
            ]["Aliases"]
        }

        mapping_method = (
            "Exact"
            if source_text
            in exact_alias_texts
            else "Normalised Alias"
        )

        return pd.Series(
            classification_from_taxonomy_item(
                taxonomy_item=exact_matches[
                    0
                ],
                mapping_method=mapping_method,
            )
        )

    if len(exact_matches) > 1:
        return pd.Series(
            create_taxonomy_result(
                category=(
                    "Unclassified & Data Quality"
                ),
                subcategory=(
                    "Ambiguous Classification"
                ),
                mapping_status="Ambiguous",
                mapping_method=(
                    "Multiple Alias Rules"
                ),
                review_required=1,
                operational_owner=(
                    "Support Quality"
                ),
                issue_nature=(
                    "Data Quality Exception"
                ),
                improvement_theme="Review",
            )
        )

    # -------------------------------------------------------------------------
    # Description keyword rule
    # -------------------------------------------------------------------------

    description_fields = [
        "issue_description",
        "resolution_notes",
    ]

    description_parts = []

    for field in description_fields:
        if (
            field in row.index
            and pd.notna(
                row.get(field)
            )
        ):
            description_parts.append(
                str(
                    row.get(field)
                )
            )

    description_text = normalise_issue_text(
        " ".join(
            description_parts
        )
    )

    description_matches: List[
        Dict[str, Any]
    ] = []

    if description_text:
        for taxonomy_item in TAXONOMY:
            keywords = taxonomy_item[
                "DescriptionKeywords"
            ]

            if not keywords:
                continue

            keyword_match = any(
                normalise_issue_text(
                    keyword
                )
                in description_text
                for keyword in keywords
                if normalise_issue_text(
                    keyword
                )
            )

            if keyword_match:
                description_matches.append(
                    taxonomy_item
                )

    # Remove duplicate sub-category matches.
    unique_description_matches: Dict[
        str,
        Dict[str, Any],
    ] = {
        item["RK_SubCategory"]: item
        for item in description_matches
    }

    description_matches = list(
        unique_description_matches.values()
    )

    if len(description_matches) == 1:
        return pd.Series(
            classification_from_taxonomy_item(
                taxonomy_item=description_matches[
                    0
                ],
                mapping_method=(
                    "Description Rule"
                ),
            )
        )

    if len(description_matches) > 1:
        return pd.Series(
            create_taxonomy_result(
                category=(
                    "Unclassified & Data Quality"
                ),
                subcategory=(
                    "Ambiguous Classification"
                ),
                mapping_status="Ambiguous",
                mapping_method=(
                    "Multiple Description Rules"
                ),
                review_required=1,
                operational_owner=(
                    "Support Quality"
                ),
                issue_nature=(
                    "Data Quality Exception"
                ),
                improvement_theme="Review",
            )
        )

    # -------------------------------------------------------------------------
    # Unmapped category
    # -------------------------------------------------------------------------

    return pd.Series(
        create_taxonomy_result(
            category=(
                "Unclassified & Data Quality"
            ),
            subcategory="Unmapped Category",
            mapping_status="Unmapped",
            mapping_method=(
                "No Matching Rule"
            ),
            review_required=1,
            operational_owner=(
                "Data Governance"
            ),
            issue_nature=(
                "Data Quality Exception"
            ),
            improvement_theme="Review",
        )
    )


# =============================================================================
# 7. APPLY CLASSIFICATION
# =============================================================================

classification_fields = (
    tickets_holidays_categorised_df.apply(
        classify_ticket,
        axis=1,
    )
)


tickets_holidays_categorised_df = (
    pd.concat(
        [
            tickets_holidays_categorised_df,
            classification_fields,
        ],
        axis=1,
    )
)


# =============================================================================
# 8. OPTIONAL MANUAL REVIEW OVERRIDE
# =============================================================================

# This field allows a later reviewed mapping to be identified explicitly.
# The code does not automatically classify a record as manually reviewed.

tickets_holidays_categorised_df[
    "RK_CategoryManualOverride"
] = pd.NA


tickets_holidays_categorised_df[
    "RK_SubCategoryManualOverride"
] = pd.NA


tickets_holidays_categorised_df[
    "RK_CategoryFinal"
] = (
    tickets_holidays_categorised_df[
        "RK_CategoryManualOverride"
    ]
    .fillna(
        tickets_holidays_categorised_df[
            "RK_Category"
        ]
    )
)


tickets_holidays_categorised_df[
    "RK_SubCategoryFinal"
] = (
    tickets_holidays_categorised_df[
        "RK_SubCategoryManualOverride"
    ]
    .fillna(
        tickets_holidays_categorised_df[
            "RK_SubCategory"
        ]
    )
)


manual_override_used = (
    tickets_holidays_categorised_df[
        "RK_CategoryManualOverride"
    ].notna()
    | tickets_holidays_categorised_df[
        "RK_SubCategoryManualOverride"
    ].notna()
)


tickets_holidays_categorised_df.loc[
    manual_override_used,
    "RK_CategoryMappingStatus",
] = "Mapped"


tickets_holidays_categorised_df.loc[
    manual_override_used,
    "RK_CategoryMappingMethod",
] = "Manual"


tickets_holidays_categorised_df.loc[
    manual_override_used,
    "RK_CategoryReviewRequired",
] = 0


# =============================================================================
# 9. VALIDATE THE TAXONOMY
# =============================================================================

expected_categories: Set[str] = {
    "Account & Access",
    "Billing, Payments & Subscription",
    "Product Reliability & Defects",
    "Product Improvement",
    "Security & Privacy",
    "Unclassified & Data Quality",
}


expected_subcategories: Set[str] = {
    "Login Issue",
    "Account Suspension",
    "Payment Problem",
    "Refund Request",
    "Subscription Cancellation",
    "Bug Report",
    "Performance Issue",
    "Data Sync Issue",
    "Feature Request",
    "Security Concern",
    "Missing Category",
    "Unmapped Category",
    "Ambiguous Classification",
    "Manual Review Required",
}


configured_categories = {
    item["RK_Category"]
    for item in TAXONOMY
}


configured_subcategories = {
    item["RK_SubCategory"]
    for item in TAXONOMY
}


if (
    configured_categories
    != expected_categories
):
    raise ValueError(
        "The configured category list does not "
        "contain exactly the required six categories."
    )


if (
    configured_subcategories
    != expected_subcategories
):
    raise ValueError(
        "The configured sub-category list does not "
        "contain exactly the required fourteen "
        "sub-categories."
    )


if len(
    tickets_holidays_categorised_df
) != source_row_count:
    raise ValueError(
        "Classification changed the ticket row count. "
        f"Before: {source_row_count:,}; "
        f"after: "
        f"{len(tickets_holidays_categorised_df):,}."
    )


# Confirm the source category was unchanged.
if not (
    tickets_holidays_categorised_df[
        "category"
    ]
    .reset_index(drop=True)
    .equals(
        original_category_snapshot
        .reset_index(drop=True)
    )
):
    raise ValueError(
        "The original category field was modified. "
        "The classified dataframe has not been accepted."
    )


required_new_fields = [
    "RK_Category",
    "RK_SubCategory",
    "RK_CategoryMappingStatus",
    "RK_CategoryMappingMethod",
    "RK_CategoryReviewRequired",
    "RK_OperationalOwner",
    "RK_IssueNature",
    "RK_ImprovementTheme",
]


missing_new_fields = [
    field
    for field in required_new_fields
    if field
    not in tickets_holidays_categorised_df.columns
]


if missing_new_fields:
    raise ValueError(
        "Classification output is missing fields: "
        f"{missing_new_fields}"
    )


invalid_mapping_status = (
    ~tickets_holidays_categorised_df[
        "RK_CategoryMappingStatus"
    ].isin(
        {
            "Mapped",
            "Missing",
            "Unmapped",
            "Ambiguous",
        }
    )
)


if invalid_mapping_status.any():
    raise ValueError(
        "Unexpected RK_CategoryMappingStatus values "
        "were created."
    )


invalid_mapping_method = (
    ~tickets_holidays_categorised_df[
        "RK_CategoryMappingMethod"
    ].isin(
        {
            "Exact",
            "Normalised Alias",
            "Description Rule",
            "Manual",
            "No Source Value",
            "No Matching Rule",
            "Multiple Alias Rules",
            "Multiple Description Rules",
        }
    )
)


if invalid_mapping_method.any():
    raise ValueError(
        "Unexpected RK_CategoryMappingMethod values "
        "were created."
    )


# =============================================================================
# 10. TAXONOMY DICTIONARY
# =============================================================================

taxonomy_dictionary_rows = []


for taxonomy_item in TAXONOMY:
    taxonomy_dictionary_rows.append(
        {
            "RK_Category": taxonomy_item[
                "RK_Category"
            ],
            "RK_SubCategory": taxonomy_item[
                "RK_SubCategory"
            ],
            "RK_OperationalOwner": taxonomy_item[
                "RK_OperationalOwner"
            ],
            "RK_IssueNature": taxonomy_item[
                "RK_IssueNature"
            ],
            "RK_ImprovementTheme": taxonomy_item[
                "RK_ImprovementTheme"
            ],
            "ApprovedAliases": " | ".join(
                sorted(
                    taxonomy_item[
                        "Aliases"
                    ]
                )
            ),
            "DescriptionKeywords": " | ".join(
                sorted(
                    taxonomy_item[
                        "DescriptionKeywords"
                    ]
                )
            ),
        }
    )


rk_taxonomy_dictionary_df = (
    pd.DataFrame(
        taxonomy_dictionary_rows
    )
    .sort_values(
        [
            "RK_Category",
            "RK_SubCategory",
        ]
    )
    .reset_index(drop=True)
)


# =============================================================================
# 11. MAPPING SUMMARY
# =============================================================================

ticket_count_field = (
    "ticket_id"
    if "ticket_id"
    in tickets_holidays_categorised_df.columns
    else "category"
)


rk_category_mapping_summary_df = (
    tickets_holidays_categorised_df
    .groupby(
        [
            "RK_CategoryMappingStatus",
            "RK_Category",
            "RK_SubCategory",
            "RK_IssueNature",
            "RK_OperationalOwner",
            "RK_ImprovementTheme",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        TicketCount=(
            ticket_count_field,
            "size",
        )
    )
)


rk_category_mapping_summary_df[
    "TicketPercentage"
] = (
    rk_category_mapping_summary_df[
        "TicketCount"
    ]
    / len(
        tickets_holidays_categorised_df
    )
    * 100
)


rk_category_mapping_summary_df = (
    rk_category_mapping_summary_df
    .sort_values(
        "TicketCount",
        ascending=False,
    )
    .reset_index(drop=True)
)


# =============================================================================
# 12. UNMAPPED AND REVIEW QUEUE
# =============================================================================

review_required_mask = (
    tickets_holidays_categorised_df[
        "RK_CategoryReviewRequired"
    ].eq(1)
)


review_columns = [
    field
    for field in [
        "ticket_id",
        "category",
        "RK_CategoryNormalised",
        "issue_description",
        "service_area",
        "team",
        "RK_CategoryMappingStatus",
        "RK_CategoryMappingMethod",
        "RK_Category",
        "RK_SubCategory",
    ]
    if field
    in tickets_holidays_categorised_df.columns
]


rk_category_review_queue_df = (
    tickets_holidays_categorised_df.loc[
        review_required_mask,
        review_columns,
    ]
    .copy()
    .reset_index(drop=True)
)


rk_unmapped_categories_df = (
    tickets_holidays_categorised_df.loc[
        tickets_holidays_categorised_df[
            "RK_CategoryMappingStatus"
        ].isin(
            {
                "Unmapped",
                "Ambiguous",
                "Missing",
            }
        ),
        [
            "category",
            "RK_CategoryNormalised",
            "RK_CategoryMappingStatus",
            "RK_CategoryMappingMethod",
        ],
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "TicketCount"
    )
    .reset_index()
    .sort_values(
        "TicketCount",
        ascending=False,
    )
    .reset_index(drop=True)
)


# =============================================================================
# 13. PRINT RESULTS
# =============================================================================

mapped_count = int(
    tickets_holidays_categorised_df[
        "RK_CategoryMappingStatus"
    ].eq("Mapped").sum()
)


review_count = int(
    tickets_holidays_categorised_df[
        "RK_CategoryReviewRequired"
    ].sum()
)


mapping_coverage = (
    mapped_count
    / source_row_count
    if source_row_count
    else 0.0
)


print("=" * 72)
print("OPERATIONAL SUPPORT TAXONOMY ADDED")
print("=" * 72)

print(
    f"Source rows: "
    f"{source_row_count:,}"
)

print(
    f"Output rows: "
    f"{len(tickets_holidays_categorised_df):,}"
)

print(
    f"Configured categories: "
    f"{len(configured_categories)}"
)

print(
    f"Configured sub-categories: "
    f"{len(configured_subcategories)}"
)

print(
    f"Mapped tickets: "
    f"{mapped_count:,}"
)

print(
    f"Mapping coverage: "
    f"{mapping_coverage:.1%}"
)

print(
    f"Tickets requiring review: "
    f"{review_count:,}"
)

print(
    "Original category field unchanged: Yes"
)

print()
print(
    "New dataframe: "
    "tickets_holidays_categorised_df"
)

print(
    f"Shape: "
    f"{tickets_holidays_categorised_df.shape}"
)




# =============================================================================
# 14. DISPLAY SUMMARIES
# =============================================================================

display(
    rk_category_mapping_summary_df
)


if not rk_unmapped_categories_df.empty:
    print()
    print(
        "Unmapped, missing or ambiguous "
        "source categories:"
    )

    display(
        rk_unmapped_categories_df
    )




OPERATIONAL SUPPORT TAXONOMY ADDED
Source rows: 100,851
Output rows: 100,851
Configured categories: 6
Configured sub-categories: 14
Mapped tickets: 100,851
Mapping coverage: 100.0%
Tickets requiring review: 0
Original category field unchanged: Yes

New dataframe: tickets_holidays_categorised_df
Shape: (100851, 75)


,RK_CategoryMappingStatus,RK_Category,RK_SubCategory,RK_IssueNature,RK_OperationalOwner,RK_ImprovementTheme,TicketCount,TicketPercentage
0,Mapped,Product Improvement,Feature Request,Request,Product Management,Review,10228,10.141694
1,Mapped,Product Reliability & Defects,Bug Report,Incident,Product / Engineering,Product Fix,10130,10.044521
2,Mapped,Account & Access,Login Issue,Incident,Support / Identity,Automation,10128,10.042538
3,Mapped,"Billing, Payments & Subscription",Subscription Cancellation,Commercial,Subscription Operations,Automation,10108,10.022707
4,Mapped,"Billing, Payments & Subscription",Refund Request,Commercial,Billing / Credit,Process Fix,10094,10.008825
5,Mapped,Product Reliability & Defects,Performance Issue,Incident,Platform / Engineering,Product Fix,10092,10.006842
6,Mapped,Account & Access,Account Suspension,Incident,Credit / Account Administration,Process Fix,10077,9.991968
7,Mapped,"Billing, Payments & Subscription",Payment Problem,Commercial,Billing / Payments,Process Fix,10052,9.967179
8,Mapped,Security & Privacy,Security Concern,Security,Security / Risk,Review,9975,9.890829
9,Mapped,Product Reliability & Defects,Data Sync Issue,Incident,Integration / Data Engineering,Product Fix,9967,9.882897


## Save output file: Categorical dataset 

In [11]:
from pathlib import Path

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/"
    "TechSolve_Ticket_Data"
)

OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "rk_quality_categories_outputs"
)

OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


output_files = {
    "tickets_holidays_categorised_df": (
        OUTPUT_DIRECTORY
        / "tickets_holidays_categorised_df.csv"
    ),

    "rk_taxonomy_dictionary_df": (
        OUTPUT_DIRECTORY
        / "rk_taxonomy_dictionary_df.csv"
    ),

    "rk_category_mapping_summary_df": (
        OUTPUT_DIRECTORY
        / "rk_category_mapping_summary_df.csv"
    ),

    "rk_category_review_queue_df": (
        OUTPUT_DIRECTORY
        / "rk_category_review_queue_df.csv"
    ),

    "rk_unmapped_categories_df": (
        OUTPUT_DIRECTORY
        / "rk_unmapped_categories_df.csv"
    ),
}


tickets_holidays_categorised_df.to_csv(
    output_files[
        "tickets_holidays_categorised_df"
    ],
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)


rk_taxonomy_dictionary_df.to_csv(
    output_files[
        "rk_taxonomy_dictionary_df"
    ],
    index=False,
    encoding="utf-8-sig",
)


rk_category_mapping_summary_df.to_csv(
    output_files[
        "rk_category_mapping_summary_df"
    ],
    index=False,
    encoding="utf-8-sig",
)


rk_category_review_queue_df.to_csv(
    output_files[
        "rk_category_review_queue_df"
    ],
    index=False,
    encoding="utf-8-sig",
)


rk_unmapped_categories_df.to_csv(
    output_files[
        "rk_unmapped_categories_df"
    ],
    index=False,
    encoding="utf-8-sig",
)


print()
print("=" * 72)
print("TAXONOMY OUTPUTS SAVED")
print("=" * 72)

for output_name, output_file in output_files.items():
    print(
        f"{output_name}:"
    )

    print(
        f"  Saved: {output_file}"
    )

    print(
        f"  Exists: {output_file.exists()}"
    )

    print(
        f"  Size: "
        f"{output_file.stat().st_size:,} bytes"
    )


TAXONOMY OUTPUTS SAVED
tickets_holidays_categorised_df:
  Saved: /content/drive/MyDrive/TechSolve_Ticket_Data/rk_quality_categories_outputs/tickets_holidays_categorised_df.csv
  Exists: True
  Size: 75,905,079 bytes
rk_taxonomy_dictionary_df:
  Saved: /content/drive/MyDrive/TechSolve_Ticket_Data/rk_quality_categories_outputs/rk_taxonomy_dictionary_df.csv
  Exists: True
  Size: 3,429 bytes
rk_category_mapping_summary_df:
  Saved: /content/drive/MyDrive/TechSolve_Ticket_Data/rk_quality_categories_outputs/rk_category_mapping_summary_df.csv
  Exists: True
  Size: 1,309 bytes
rk_category_review_queue_df:
  Saved: /content/drive/MyDrive/TechSolve_Ticket_Data/rk_quality_categories_outputs/rk_category_review_queue_df.csv
  Exists: True
  Size: 157 bytes
rk_unmapped_categories_df:
  Saved: /content/drive/MyDrive/TechSolve_Ticket_Data/rk_quality_categories_outputs/rk_unmapped_categories_df.csv
  Exists: True
  Size: 96 bytes


In [13]:
tickets_holidays_categorised_df

,ticket_id,customer_id,customer_name,customer_email,company_name,account_type,customer_segment,industry,billing_contact_email,account_manager,...,RK_CategoryMappingStatus,RK_CategoryMappingMethod,RK_CategoryReviewRequired,RK_OperationalOwner,RK_IssueNature,RK_ImprovementTheme,RK_CategoryManualOverride,RK_SubCategoryManualOverride,RK_CategoryFinal,RK_SubCategoryFinal
0,1,ACC-07512,David Martin,david.martin4@hotmail.com,Central Ltd,Business,Small Business,Retail,NaN,Sarah Chen,...,Mapped,Normalised Alias,0,Security / Risk,Security,Review,<NA>,<NA>,Security & Privacy,Security Concern
1,2,ACC-07455,Patricia Martin,patricia.martin869@icloud.com,Gateway Transport,Business,Small Business,Trades,NaN,NaN,...,Mapped,Normalised Alias,0,Platform / Engineering,Incident,Product Fix,<NA>,<NA>,Product Reliability & Defects,Performance Issue
2,3,ACC-03650,John Lopez,john.lopez551@gmail.com,Pioneer Logistics,Business,Small Business,Trades,NaN,NaN,...,Mapped,Normalised Alias,0,Integration / Data Engineering,Incident,Product Fix,<NA>,<NA>,Product Reliability & Defects,Data Sync Issue
3,4,ACC-05579,Mary Martinez,mary.martinez515@company.com,Coastal Tech,Business,Small Business,Technology,NaN,NaN,...,Mapped,Normalised Alias,0,Integration / Data Engineering,Incident,Product Fix,<NA>,<NA>,Product Reliability & Defects,Data Sync Issue
4,5,ACC-07456,John Martinez,john.martinez733@hotmail.com,NaN,Residential,Individual,NaN,NaN,NaN,...,Mapped,Normalised Alias,0,Billing / Payments,Commercial,Process Fix,<NA>,<NA>,"Billing, Payments & Subscription",Payment Problem
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100846,100847,ACC-06526,Charles Johnson,charles.johnson457@yahoo.com,NaN,Residential,Individual,NaN,NaN,NaN,...,Mapped,Normalised Alias,0,Support / Identity,Incident,Automation,<NA>,<NA>,Account & Access,Login Issue
100847,100848,ACC-03650,William Brown,william.brown671@hotmail.com,Pioneer Logistics,Business,Small Business,Trades,NaN,NaN,...,Mapped,Normalised Alias,0,Billing / Credit,Commercial,Process Fix,<NA>,<NA>,"Billing, Payments & Subscription",Refund Request
100848,100849,ACC-01465,Charles Taylor,charles.taylor867@gmail.com,NaN,Residential,Individual,NaN,NaN,NaN,...,Mapped,Normalised Alias,0,Subscription Operations,Commercial,Automation,<NA>,<NA>,"Billing, Payments & Subscription",Subscription Cancellation
100849,100850,ACC-03432,Susan Thomas,susan.thomas126@gmail.com,NaN,Residential,Small Business,NaN,NaN,Rob Thompson,...,Mapped,Normalised Alias,0,Product Management,Request,Review,<NA>,<NA>,Product Improvement,Feature Request


# AI Agent

Build an AI agent that can inspect your dataset and respond to natural language questions about the data. The agent should be able to answer operational questions and provide useful feedback. For example, a manager might ask about ticket trends, team performance, or problem areas.

an agent design that can inspect tickets_holidays_categorised_df dataset and respond to natural language questions. The agent should be able to answer operational questions and provide useful feedback. For example, a manager might ask about ticket trends, team performance, or problem areas. 